In [1]:
import pandas as pd
import numpy as np
import os
import json

print("Python is ready!")
print("Files found:")
print(os.listdir())

Python is ready!
Files found:
['.ipynb_checkpoints', 'applications.csv', 'biometric_punches.csv', 'data_dictionary.md', 'departments.csv', 'device_mapping.csv', 'employees.csv', 'employment_history.csv', 'holidays.csv', 'leave_requests.json', 'overtime_requests.csv', 'payroll_lines.csv', 'shifts.json', 'shift_roster.csv', 'Untitled.ipynb']


In [2]:
with open("data_dictionary.md", "r", encoding="utf-8") as f:
    print(f.read())

# Indus Weave Textiles — HCM Export Data Dictionary

Six months of exports (1 April – 30 September 2026) from the company's HCM platform.
Indus Weave Textiles is fictional. Every person, ID and number in these files is synthetic.

Plants: **Head Office** (Karachi), **Korangi Plant**, **SITE Plant**, **Faisalabad Plant**.

---

## Business rules

These rules are the "law" for this hackathon. Where the data disagrees with a rule, the rule wins.

### Employees and history
1. `employees.csv` may contain the same `emp_id` more than once. Keep the row with the latest `updated_at`.
2. `employment_history.csv` is the source of truth for department, grade and salary over time.
   The `effective_to` column in the export is **unreliable**. A record is valid from its
   `effective_from` until the day before that employee's next record starts. The latest record has no end.
3. Monthly figures (headcount, overtime, payroll cost) belong to the department the employee was in
   on the **last day of the

In [3]:
employees = pd.read_csv("employees.csv")

print("Dataset size:", employees.shape)
display(employees.head())
print("\nMissing values:")
print(employees.isnull().sum())
print("\nDuplicate rows:", employees.duplicated().sum())

Dataset size: (1316, 16)


,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at
0,E10700,Sidra Butt,F,16/08/1987,sidra.butt815@induswave.example,0305-7303046,Korangi Plant,Maintenance,Operator,G1,Permanent,17-Nov-2022,NaN,Active,14,2026-07-01 13:22:00
1,E10933,Owais Mirza,M,2007-07-27,owais.mirza894@induswave.example,0326-2458452,SITE Plant,dyeing,Technician,G3,Permanent,2016-06-18,NaN,Active,14,2026-07-01 17:53:00
2,E11260,Maryam Butt,F,2005-10-24,maryam.butt137@induswave.example,0310-8221682,Korangi Plant,Spinning,Operator,G1,Permanent,2013-08-27,NaN,Active,14,2026-07-01 14:59:00
3,E10151,Hira Javed,F,1997-01-28,hira.javed889@induswave.example,0302-1666404,Faisalabad Plant,Stitching,Senior Supervisor,G5,Permanent,2015-10-10,NaN,Active,20,2026-07-01 13:30:00
4,E10879,Sidra Iqbal,F,1988-09-10,sidra.iqbal863@induswave.example,0313-5507671,Faisalabad Plant,Stitching,Senior Supervisor,G5,Permanent,2019-04-02,NaN,Active,20,2026-07-01 10:43:00



Missing values:
emp_id                         0
full_name                      0
gender                         0
date_of_birth                  0
email                          0
phone                          0
plant                          0
department                     0
designation                    0
grade                          0
employment_type                0
hire_date                      0
separation_date             1226
status                         0
annual_leave_entitlement       0
updated_at                     0
dtype: int64

Duplicate rows: 0


In [4]:
# 1. Column names
print("COLUMNS:")
print(employees.columns.tolist())

# 2. Missing values
print("\nMISSING VALUES:")
print(employees.isnull().sum())

# 3. Duplicate employee IDs
print("\nDUPLICATE EMPLOYEE IDs:")
print(employees["emp_id"].duplicated().sum())

# 4. Duplicate IDs ki rows dekho
print("\nDUPLICATE EMPLOYEE RECORDS:")
display(
    employees[
        employees["emp_id"].duplicated(keep=False)
    ].sort_values("emp_id").head(10)
)

COLUMNS:
['emp_id', 'full_name', 'gender', 'date_of_birth', 'email', 'phone', 'plant', 'department', 'designation', 'grade', 'employment_type', 'hire_date', 'separation_date', 'status', 'annual_leave_entitlement', 'updated_at']

MISSING VALUES:
emp_id                         0
full_name                      0
gender                         0
date_of_birth                  0
email                          0
phone                          0
plant                          0
department                     0
designation                    0
grade                          0
employment_type                0
hire_date                      0
separation_date             1226
status                         0
annual_leave_entitlement       0
updated_at                     0
dtype: int64

DUPLICATE EMPLOYEE IDs:
50

DUPLICATE EMPLOYEE RECORDS:


,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at
21,E10026,Nadeem Soomro,M,1976-09-12,nadeem.soomro896@induswave.example,0307-9559808,SITE Plant,MAINTENANCE,Operator,G1,Permanent,18/09/2013,NaN,Active,14,2026-07-01 18:42:00
989,E10026,Nadeem Soomro,M,1976-09-12,nadeem.soomro896@induswave.example,03079559808,SITE Plant,Maintenance,Operator,G1,Permanent,18/09/2013,NaN,Active,14,2026-05-07 18:42:00
468,E10142,Usman Baloch,M,1986-08-22,usman.baloch502@induswave.example,0319-9448050,Faisalabad Plant,stitching,Operator,G1,Contract,2014-10-14,NaN,Active,14,2026-07-01 16:18:00
766,E10142,USMAN BALOCH,M,1986-08-22,usman.baloch502@induswave.example,0319-9448050,Faisalabad Plant,Stitching,Operator,G1,Contract,2014-10-14,NaN,Active,14,2026-01-27 16:18:00
187,E10162,ASIF MIRZA,M,2004-12-21,asif.mirza67@induswave.example,0340-4307056,SITE Plant,Dyeing,Operator,G1,Permanent,2014-03-15,NaN,Active,14,2025-10-28 11:15:00
803,E10162,Asif Mirza,M,2004-12-21,asif.mirza67@induswave.example,0340-4307056,SITE Plant,DYEING,Operator,G1,Permanent,2014-03-15,NaN,Active,14,2026-07-01 11:15:00
933,E10191,FAISAL RAZA,M,10-Nov-1989,faisal.raza236@induswave.example,0340-1765349,Faisalabad Plant,Packing,Operator,G1,Permanent,2018-05-01,NaN,Active,14,2026-01-22 16:45:00
972,E10191,Faisal Raza,M,10-Nov-1989,faisal.raza236@induswave.example,0340-1765349,Faisalabad Plant,Packing,Operator,G1,Permanent,2018-05-01,NaN,Active,14,2026-07-01 16:45:00
16,E10208,Nadeem Baloch,M,1996-10-25,nadeem.baloch947@induswave.example,0346-7439169,SITE Plant,Finishing,Operator,G1,Permanent,22/03/2013,NaN,Active,14,2026-07-01 10:20:00
432,E10208,NADEEM BALOCH,M,1996-10-25,nadeem.baloch947@induswave.example,0346-7439169,SITE Plant,Finishing,Operator,G1,Permanent,22/03/2013,NaN,Active,14,2025-09-12 10:20:00


In [5]:
print(employees["updated_at"].head(10))

print("\nUpdated_at data type:")
print(employees["updated_at"].dtype)

print("\nMissing updated_at:")
print(employees["updated_at"].isnull().sum())


0    2026-07-01 13:22:00
1    2026-07-01 17:53:00
2    2026-07-01 14:59:00
3    2026-07-01 13:30:00
4    2026-07-01 10:43:00
5    2026-07-01 12:48:00
6    2026-07-01 12:27:00
7    2026-04-16 13:14:00
8    2026-07-01 15:51:00
9    2026-07-01 13:13:00
Name: updated_at, dtype: str

Updated_at data type:
str

Missing updated_at:
0


In [6]:
# Original data ko safe rakho
employees_raw = employees.copy()

# updated_at ko datetime mein convert karo
employees["updated_at"] = pd.to_datetime(
    employees["updated_at"], errors="coerce"
)

# Latest updated_at wali row rakho
employees_clean = (
    employees.sort_values("updated_at")
    .drop_duplicates(subset="emp_id", keep="last")
    .copy()
)

print("Original rows:", len(employees_raw))
print("Clean rows:", len(employees_clean))
print("Duplicate IDs remaining:",
      employees_clean["emp_id"].duplicated().sum())

Original rows: 1316
Clean rows: 1266
Duplicate IDs remaining: 0


In [7]:
# Text ke aage/peeche ke spaces remove karo
text_cols = employees_clean.select_dtypes(
    include="object"
).columns

for col in text_cols:
    employees_clean[col] = employees_clean[col].str.strip()

# Names aur categories ka format consistent karo
for col in ["full_name", "plant", "department",
            "designation", "employment_type", "status"]:
    if col in employees_clean.columns:
        employees_clean[col] = (
            employees_clean[col].str.title()
        )

# Email lowercase mein
employees_clean["email"] = employees_clean["email"].str.lower()

# Gender aur grade consistent rakho
for col in ["gender", "grade"]:
    if col in employees_clean.columns:
        employees_clean[col] = (
            employees_clean[col].str.upper()
        )

print(employees_clean[
    ["emp_id", "full_name", "plant", "department"]
].head(10))

      emp_id      full_name             plant       department
335   E10112   Rizwan Mirza  Faisalabad Plant          Packing
293   E10899   Tariq Akhtar       Head Office          Finance
433   E11056   Hamza Hashmi  Faisalabad Plant          Packing
1069  E11172    Kamran Khan     Korangi Plant         Spinning
555   E10144   Waqas Bhatti       Head Office  Human Resources
1253  E11137  Rizwan Ansari     Korangi Plant          Weaving
7     E11025   Owais Ansari     Korangi Plant          Weaving
313   E10130      Hina Raza        Site Plant  Quality Control
1314  E10795      Ali Rizvi     Korangi Plant          Weaving
328   E10125    Fahad Malik  Faisalabad Plant        Stitching


C:\Users\dell\AppData\Local\Temp\ipykernel_49200\1201043752.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = employees_clean.select_dtypes(


In [8]:
# Text columns identify karo
text_cols = employees_clean.select_dtypes(
    include=["object", "str"]
).columns

# Extra spaces remove karo
for col in text_cols:
    employees_clean[col] = (
        employees_clean[col].str.strip()
    )

# Department aur plant ke names consistent karo
for col in ["department", "plant"]:
    if col in employees_clean.columns:
        employees_clean[col] = (
            employees_clean[col].str.title()
        )

print("Text cleaning complete!")
display(
    employees_clean[
        ["emp_id", "full_name", "plant", "department"]
    ].head(10)
)

Text cleaning complete!


,emp_id,full_name,plant,department
335,E10112,Rizwan Mirza,Faisalabad Plant,Packing
293,E10899,Tariq Akhtar,Head Office,Finance
433,E11056,Hamza Hashmi,Faisalabad Plant,Packing
1069,E11172,Kamran Khan,Korangi Plant,Spinning
555,E10144,Waqas Bhatti,Head Office,Human Resources
1253,E11137,Rizwan Ansari,Korangi Plant,Weaving
7,E11025,Owais Ansari,Korangi Plant,Weaving
313,E10130,Hina Raza,Site Plant,Quality Control
1314,E10795,Ali Rizvi,Korangi Plant,Weaving
328,E10125,Fahad Malik,Faisalabad Plant,Stitching


In [9]:
employees_clean.to_csv(
    "employees_clean.csv", index=False
)

print("File saved successfully!")
print("Total employees:", len(employees_clean))

File saved successfully!
Total employees: 1266


In [10]:
# Biometric attendance data load karo
biometric = pd.read_csv("biometric_punches.csv")

print("Dataset size:", biometric.shape)

print("\nColumn names:")
print(biometric.columns.tolist())

print("\nFirst 5 records:")
display(biometric.head())

print("\nMissing values:")
print(biometric.isnull().sum())

print("\nDuplicate rows:", biometric.duplicated().sum())

Dataset size: (345280, 4)

Column names:
['punch_id', 'device_id', 'device_user_id', 'punch_time']

First 5 records:


,punch_id,device_id,device_user_id,punch_time
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33



Missing values:
punch_id          0
device_id         0
device_user_id    0
punch_time        0
dtype: int64

Duplicate rows: 0


In [11]:
# Original biometric data safe rakho
biometric_raw = biometric.copy()

# Punch time ko datetime mein convert karo
biometric["punch_time"] = pd.to_datetime(
    biometric["punch_time"],
    errors="coerce"
)

# Basic validation
print("Original rows:", len(biometric_raw))

print("Exact duplicate rows:",
      biometric.duplicated().sum())

print("Duplicate punch IDs:",
      biometric["punch_id"].duplicated().sum())

print("Invalid punch times:",
      biometric["punch_time"].isna().sum())

print("Unique devices:",
      biometric["device_id"].nunique())

print("Unique device users:",
      biometric["device_user_id"].nunique())

Original rows: 345280
Exact duplicate rows: 0
Duplicate punch IDs: 0
Invalid punch times: 0
Unique devices: 4
Unique device users: 1299


In [12]:
# Device mapping file load karo
mapping = pd.read_csv("device_mapping.csv")

print("Mapping dataset size:", mapping.shape)

print("\nColumn names:")
print(mapping.columns.tolist())

print("\nFirst 10 records:")
display(mapping.head(10))

print("\nMissing values:")
print(mapping.isnull().sum())

print("\nDuplicate rows:", mapping.duplicated().sum())

Mapping dataset size: (1260, 4)

Column names:
['device_id', 'device_user_id', 'emp_id', 'enrolled_on']

First 10 records:


,device_id,device_user_id,emp_id,enrolled_on
0,DEV-SITE-01,16779,E10250,2024-01-01
1,DEV-FSD-01,10465,E10591,2024-01-01
2,DEV-KOR-01,36101,E10916,2024-01-01
3,DEV-FSD-01,7339,E10865,2024-01-01
4,DEV-KOR-01,26390,E10683,2025-01-31
5,DEV-SITE-01,96855,E11233,2026-05-27
6,DEV-FSD-01,61672,E10100,2024-01-01
7,DEV-FSD-01,77417,E10489,2026-05-25
8,DEV-HO-01,4381,E10451,2024-01-01
9,DEV-KOR-01,88699,E10563,2024-01-01



Missing values:
device_id         0
device_user_id    0
emp_id            0
enrolled_on       0
dtype: int64

Duplicate rows: 0


In [14]:
import pandas as pd

device_mapping = pd.read_csv("device_mapping.csv")

print(device_mapping.shape)
print(device_mapping.columns.tolist())

(1260, 4)
['device_id', 'device_user_id', 'emp_id', 'enrolled_on']


In [15]:
print("Missing values:")
print(device_mapping.isnull().sum())

print("\nDuplicate rows:")
print(device_mapping.duplicated().sum())

print("\nFirst 5 records:")
display(device_mapping.head())

Missing values:
device_id         0
device_user_id    0
emp_id            0
enrolled_on       0
dtype: int64

Duplicate rows:
0

First 5 records:


,device_id,device_user_id,emp_id,enrolled_on
0,DEV-SITE-01,16779,E10250,2024-01-01
1,DEV-FSD-01,10465,E10591,2024-01-01
2,DEV-KOR-01,36101,E10916,2024-01-01
3,DEV-FSD-01,7339,E10865,2024-01-01
4,DEV-KOR-01,26390,E10683,2025-01-31


In [16]:
# Device mapping ke employee IDs verify karo

missing_employees = device_mapping.loc[
    ~device_mapping["emp_id"].isin(
        employees_clean["emp_id"]
    )
]

print("Unmatched employee records:", len(missing_employees))

display(missing_employees.head(10))

Unmatched employee records: 0


,device_id,device_user_id,emp_id,enrolled_on


In [17]:
import pandas as pd

biometric = pd.read_csv("biometric_punches.csv")

print("Dataset size:", biometric.shape)

print("\nColumns:")
print(biometric.columns.tolist())

print("\nMissing values:")
print(biometric.isnull().sum())

print("\nDuplicate rows:",
      biometric.duplicated().sum())

print("\nDuplicate punch IDs:",
      biometric["punch_id"].duplicated().sum())

print("\nFirst 5 records:")
display(biometric.head())


Dataset size: (345280, 4)

Columns:
['punch_id', 'device_id', 'device_user_id', 'punch_time']

Missing values:
punch_id          0
device_id         0
device_user_id    0
punch_time        0
dtype: int64

Duplicate rows: 0

Duplicate punch IDs: 0

First 5 records:


,punch_id,device_id,device_user_id,punch_time
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33


In [18]:
biometric["punch_time"] = pd.to_datetime(
    biometric["punch_time"],
    errors="coerce"
)

print("Invalid punch times:",
      biometric["punch_time"].isna().sum())

print("Earliest punch:",
      biometric["punch_time"].min())

print("Latest punch:",
      biometric["punch_time"].max())

Invalid punch times: 0
Earliest punch: 2026-04-01 05:35:12
Latest punch: 2026-10-01 10:24:52


In [19]:
# Biometric records ko device mapping se match karo

biometric["device_user_id"] = biometric["device_user_id"].astype(str)
device_mapping["device_user_id"] = device_mapping["device_user_id"].astype(str)

attendance = biometric.merge(
    device_mapping[
        ["device_user_id", "emp_id", "device_id"]
    ],
    on="device_user_id",
    how="left",
    suffixes=("", "_mapping")
)

print("Total attendance records:", len(attendance))
print("Unmatched attendance records:",
      attendance["emp_id"].isna().sum())

display(attendance.head())

Total attendance records: 345280
Unmatched attendance records: 6784


,punch_id,device_id,device_user_id,punch_time,emp_id,device_id_mapping
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12,E10543,DEV-KOR-01
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14,E10568,DEV-SITE-01
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30,E10384,DEV-KOR-01
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33,E11112,DEV-FSD-01
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33,E10880,DEV-SITE-01


In [20]:
print("Attendance rows:", len(attendance))

print("Duplicate punch IDs:",
      attendance["punch_id"].duplicated().sum())

print("Missing employee IDs:",
      attendance["emp_id"].isna().sum())

print("Missing punch times:",
      attendance["punch_time"].isna().sum())

Attendance rows: 345280
Duplicate punch IDs: 0
Missing employee IDs: 6784
Missing punch times: 0


In [21]:
attendance["punch_date"] = attendance["punch_time"].dt.date

daily_summary = (
    attendance.groupby("punch_date")
    .agg(
        total_punches=("punch_id", "count"),
        unique_employees=("emp_id", "nunique")
    )
    .reset_index()
)

display(daily_summary.head(10))

,punch_date,total_punches,unique_employees
0,2026-04-01,2088,1134
1,2026-04-02,2369,1137
2,2026-04-03,2356,1123
3,2026-04-04,2323,1106
4,2026-04-05,364,324
5,2026-04-06,2008,1119
6,2026-04-07,2363,1134
7,2026-04-08,2321,1123
8,2026-04-09,2313,1117
9,2026-04-10,2379,1130


In [22]:
attendance.to_csv(
    "attendance_clean.csv",
    index=False
)

daily_summary.to_csv(
    "daily_attendance_summary.csv",
    index=False
)

print("Files saved successfully!")

Files saved successfully!


In [23]:
%pip install sqlalchemy pymysql

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 635.4 kB/s eta 0:00:04
   ------------ --------------------------- 0.8/2.4 MB 834.7 kB/s eta 0:00:02
   --------------------- ------------------ 1.3/2.4 MB 1.1 MB/s eta 0:00:02
   --------------------- ------------------ 1.3/2.4 MB 1.1 MB/s eta 0:00:02
   ------------------------------ --------- 1.8/2.4 MB 1.1 MB/s eta 0:0


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
 pip install sqlalchemy pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

server = r"(localdb)\MSSQLLocalDB"
database = "peopleops_hackathon"

# Windows Authentication
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string}
)

engine = create_engine(connection_url)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT DB_NAME()"))
        print("Connected successfully!")
        print("Database:", result.scalar())

except Exception as e:
    print("Connection failed:")
    print(e)

finally:
    engine.dispose()

Connected successfully!
Database: peopleops_hackathon


In [34]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    """))

    for row in result:
        print(row[0])

In [35]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT DB_NAME(), COUNT(*)
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    """))

    print(result.fetchone())

('peopleops_hackathon', 0)


In [36]:
from sqlalchemy import text

with engine.connect() as conn:
    print("Database:", conn.execute(
        text("SELECT DB_NAME()")
    ).scalar())

    result = conn.execute(text("""
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    """))

    tables = result.fetchall()

    if tables:
        print("Tables found:")
        for row in tables:
            print(row[0] + "." + row[1])
    else:
        print("No tables found in this database.")

Database: peopleops_hackathon
No tables found in this database.


In [37]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Folder ke andar files:")

for item in Path.cwd().iterdir():
    print(item.name)

Current folder: C:\Users\dell\Desktop\data-hackathon-main
Folder ke andar files:
.ipynb_checkpoints
applications.csv
attendance_clean.csv
biometric_punches.csv
daily_attendance_summary.csv
data_dictionary.md
departments.csv
device_mapping.csv
employees.csv
employees_clean.csv
employment_history.csv
holidays.csv
leave_requests.json
overtime_requests.csv
payroll_lines.csv
shifts.json
shift_roster.csv
Untitled.ipynb


In [38]:
from pathlib import Path

folder = Path.cwd()

csv_files = list(folder.glob("*.csv"))
json_files = list(folder.glob("*.json"))

print("CSV files:", len(csv_files))
print("JSON files:", len(json_files))

for file in csv_files:
    print(file.name)

CSV files: 13
JSON files: 2
applications.csv
attendance_clean.csv
biometric_punches.csv
daily_attendance_summary.csv
departments.csv
device_mapping.csv
employees.csv
employees_clean.csv
employment_history.csv
holidays.csv
overtime_requests.csv
payroll_lines.csv
shift_roster.csv


In [39]:
import sqlite3
import pandas as pd
from pathlib import Path

folder = Path.cwd()
db_path = folder / "peopleops_hackathon.db"

csv_files = [
    "employees.csv",
    "employment_history.csv",
    "departments.csv",
    "shift_roster.csv",
    "device_mapping.csv",
    "biometric_punches.csv",
    "holidays.csv",
    "overtime_requests.csv",
    "payroll_lines.csv",
    "applications.csv"
]

conn = sqlite3.connect(db_path)

for filename in csv_files:
    file_path = folder / filename

    df = pd.read_csv(file_path, dtype=str, keep_default_na=False)

    # Raw data ke liye tracking columns
    df["_source_file"] = filename
    df["_loaded_at"] = pd.Timestamp.now(tz="UTC").isoformat()
    df["_row_hash"] = pd.util.hash_pandas_object(
        df.astype(str), index=False
    ).astype(str)

    table_name = "raw_" + Path(filename).stem

    df.to_sql(table_name, conn, if_exists="replace", index=False)

    print(f"{table_name}: {len(df)} rows loaded")

conn.close()

print("\nDatabase created:", db_path)

raw_employees: 1316 rows loaded
raw_employment_history: 3686 rows loaded
raw_departments: 25 rows loaded
raw_shift_roster: 36402 rows loaded
raw_device_mapping: 1260 rows loaded
raw_biometric_punches: 345280 rows loaded
raw_holidays: 12 rows loaded
raw_overtime_requests: 22098 rows loaded
raw_payroll_lines: 53617 rows loaded
raw_applications: 3835 rows loaded

Database created: C:\Users\dell\Desktop\data-hackathon-main\peopleops_hackathon.db


In [40]:
import sqlite3

conn = sqlite3.connect("peopleops_hackathon.db")

tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()

for table in tables:
    print(table[0])

conn.close()

raw_applications
raw_biometric_punches
raw_departments
raw_device_mapping
raw_employees
raw_employment_history
raw_holidays
raw_overtime_requests
raw_payroll_lines
raw_shift_roster


In [41]:
import sqlite3
import json
from pathlib import Path
import pandas as pd

folder = Path.cwd()
conn = sqlite3.connect("peopleops_hackathon.db")

for filename in ["leave_requests.json", "shifts.json"]:
    with open(folder / filename, "r", encoding="utf-8") as f:
        content = json.load(f)

    # JSON ko original text ke taur par preserve karna
    with open(folder / filename, "r", encoding="utf-8") as f:
        raw_text = f.read()

    df = pd.DataFrame([{
        "source_file": filename,
        "raw_content": raw_text,
        "loaded_at": pd.Timestamp.now(tz="UTC").isoformat()
    }])

    df.to_sql("raw_json_files", conn, if_exists="append", index=False)

    print(filename, "loaded successfully")

conn.close()

leave_requests.json loaded successfully
shifts.json loaded successfully


In [42]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

display(tables)

print("Employees ki total rows:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM raw_employees",
    conn
))

print("JSON files:")
display(pd.read_sql_query(
    "SELECT source_file FROM raw_json_files",
    conn
))

conn.close()

,name
0,raw_applications
1,raw_biometric_punches
2,raw_departments
3,raw_device_mapping
4,raw_employees
5,raw_employment_history
6,raw_holidays
7,raw_json_files
8,raw_overtime_requests
9,raw_payroll_lines


Employees ki total rows:


,total
0,1316


JSON files:


,source_file
0,leave_requests.json
1,shifts.json


In [43]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

display(pd.read_sql_query(
    "SELECT source_file FROM raw_json_files",
    conn
))

print("Employees ki total rows:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM raw_employees",
    conn
))

print("Biometric punches ki total rows:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM raw_biometric_punches",
    conn
))

conn.close()

,source_file
0,leave_requests.json
1,shifts.json


Employees ki total rows:


,total_rows
0,1316


Biometric punches ki total rows:


,total_rows
0,345280


In [44]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

employees = pd.read_sql_query("""
    SELECT *
    FROM raw_employees
    LIMIT 5
""", conn)

display(employees)

conn.close()

,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at,_source_file,_loaded_at,_row_hash
0,E10700,Sidra Butt,F,16/08/1987,sidra.butt815@induswave.example,0305-7303046,Korangi Plant,Maintenance,Operator,G1,Permanent,17-Nov-2022,,Active,14,2026-07-01 13:22:00,employees.csv,2026-10-24T20:30:27.189341+00:00,10550746039428947528
1,E10933,Owais Mirza,M,2007-07-27,owais.mirza894@induswave.example,0326-2458452,SITE Plant,dyeing,Technician,G3,Permanent,2016-06-18,,Active,14,2026-07-01 17:53:00,employees.csv,2026-10-24T20:30:27.189341+00:00,16389094467490310068
2,E11260,Maryam Butt,F,2005-10-24,maryam.butt137@induswave.example,0310-8221682,Korangi Plant,Spinning,Operator,G1,Permanent,2013-08-27,,Active,14,2026-07-01 14:59:00,employees.csv,2026-10-24T20:30:27.189341+00:00,7099927023713425526
3,E10151,Hira Javed,F,1997-01-28,hira.javed889@induswave.example,0302-1666404,Faisalabad Plant,Stitching,Senior Supervisor,G5,Permanent,2015-10-10,,Active,20,2026-07-01 13:30:00,employees.csv,2026-10-24T20:30:27.189341+00:00,1307191721952164066
4,E10879,Sidra Iqbal,F,1988-09-10,sidra.iqbal863@induswave.example,0313-5507671,Faisalabad Plant,Stitching,Senior Supervisor,G5,Permanent,2019-04-02,,Active,20,2026-07-01 10:43:00,employees.csv,2026-10-24T20:30:27.189341+00:00,983966994985610194


In [45]:
conn = sqlite3.connect("peopleops_hackathon.db")

columns = pd.read_sql_query("""
    PRAGMA table_info(raw_employees)
""", conn)

display(columns[["name", "type"]])

conn.close()

,name,type
0,emp_id,TEXT
1,full_name,TEXT
2,gender,TEXT
3,date_of_birth,TEXT
4,email,TEXT
5,phone,TEXT
6,plant,TEXT
7,department,TEXT
8,designation,TEXT
9,grade,TEXT


In [46]:
conn = sqlite3.connect("peopleops_hackathon.db")

duplicates = pd.read_sql_query("""
    SELECT
        emp_id,
        COUNT(*) AS record_count
    FROM raw_employees
    GROUP BY emp_id
    HAVING COUNT(*) > 1
    ORDER BY record_count DESC
""", conn)

print("Duplicate employee IDs:", len(duplicates))
display(duplicates.head(10))

conn.close()

Duplicate employee IDs: 50


,emp_id,record_count
0,E11262,2
1,E11242,2
2,E11225,2
3,E11164,2
4,E11153,2
5,E11130,2
6,E11129,2
7,E11085,2
8,E11075,2
9,E11071,2


In [47]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query("""
    SELECT *
    FROM raw_employees
    WHERE emp_id = 'E11262'
    ORDER BY updated_at DESC
""", conn)

display(df)

conn.close()

,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at,_source_file,_loaded_at,_row_hash
0,E11262,Salman Raza,M,1993-11-01,salman.raza209@induswave.example,0327-8827238,Korangi Plant,WEAVING,Operator,G1,Contract,2026-06-15,,Active,14,2026-07-01 12:16:00,employees.csv,2026-10-24T20:30:27.189341+00:00,18311897161893729207
1,E11262,Salman Raza,M,1993-11-01,salman.raza209@induswave.example,03278827238,Korangi Plant,Weaving,Operator,G1,Contract,2026-06-15,,Active,14,2026-01-05 12:16:00,employees.csv,2026-10-24T20:30:27.189341+00:00,4780768316861255350


In [50]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

latest_employee = pd.read_sql_query("""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY emp_id
                   ORDER BY updated_at DESC
               ) AS rn
        FROM raw_employees
    )
    WHERE rn = 1
""", conn)

original_rows = pd.read_sql_query(
    "SELECT COUNT(*) AS total FROM raw_employees",
    conn
).iloc[0, 0]

print("Original rows:", original_rows)
print("Unique employees after deduplication:", len(latest_employee))

display(latest_employee.head())

conn.close()

Original rows: 1316
Unique employees after deduplication: 1266


,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at,_source_file,_loaded_at,_row_hash,rn
0,E10001,Tariq Jatoi,M,13/06/1999,tariq.jatoi217@induswave.example,0331-1955353,Korangi Plant,Weaving,Operator,G1,Permanent,19/04/2019,2026-06-02,Separated,14,2026-06-02 12:25:00,employees.csv,2026-10-24T20:30:27.189341+00:00,8017033611084261128,1
1,E10002,Hamza Butt,M,1986-04-07,hamza.butt922@induswave.example,0347-4313277,Faisalabad Plant,cutting,Operator,G1,Contract,2016-02-15,,Active,14,2026-07-01 12:48:00,employees.csv,2026-10-24T20:30:27.189341+00:00,10211235729498233157,1
2,E10003,Saad Butt,M,1986-03-06,saad.butt122@induswave.example,0338-4383725,Korangi Plant,Maintenance,Senior Operator,G2,Permanent,2018-02-23,,Active,14,2026-07-01 17:47:00,employees.csv,2026-10-24T20:30:27.189341+00:00,8985980232758771058,1
3,E10004,Bilal Raza,M,23-Jul-1989,bilal.raza496@induswave.example,0317-1570993,SITE Plant,QUALITY CONTROL,Operator,G1,Permanent,2023-10-20,,Active,14,2026-07-01 15:18:00,employees.csv,2026-10-24T20:30:27.189341+00:00,9741957867171209411,1
4,E10005,Junaid Rana,M,11/01/1978,junaid.rana486@induswave.example,0313-7429960,Korangi Plant,Spinning,Operator,G1,Permanent,2020-07-25,,Active,14,2026-07-01 16:09:00,employees.csv,2026-10-24T20:30:27.189341+00:00,3821591432219566464,1


In [51]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

query = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY emp_id
               ORDER BY updated_at DESC
           ) AS rn
    FROM raw_employees
)
WHERE rn = 1
"""

# Latest record har employee ke liye
cleaned_employees = pd.read_sql_query(query, conn)

# Ranking column remove karo
cleaned_employees = cleaned_employees.drop(columns=["rn"])

# Alag staging table mein save karo
cleaned_employees.to_sql(
    "stg_employees",
    conn,
    if_exists="replace",
    index=False
)

print("Cleaned table successfully created!")
print("Rows in cleaned table:", len(cleaned_employees))

conn.close()

Cleaned table successfully created!
Rows in cleaned table: 1266


In [52]:
conn = sqlite3.connect("peopleops_hackathon.db")

check = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT emp_id) AS unique_employees
FROM stg_employees
""", conn)

display(check)

conn.close()

,total_rows,unique_employees
0,1266,1266


In [53]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Raw aur staging tables ka row count check karo
for table in ["raw_employees", "stg_employees"]:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_rows FROM {table}",
        conn
    )
    print(table)
    display(result)

conn.close()

raw_employees


,total_rows
0,1316


stg_employees


,total_rows
0,1266


In [54]:
print("Cleaned DataFrame rows:", len(cleaned_employees))
display(cleaned_employees.head())

Cleaned DataFrame rows: 1266


,emp_id,full_name,gender,date_of_birth,email,phone,plant,department,designation,grade,employment_type,hire_date,separation_date,status,annual_leave_entitlement,updated_at,_source_file,_loaded_at,_row_hash
0,E10001,Tariq Jatoi,M,13/06/1999,tariq.jatoi217@induswave.example,0331-1955353,Korangi Plant,Weaving,Operator,G1,Permanent,19/04/2019,2026-06-02,Separated,14,2026-06-02 12:25:00,employees.csv,2026-10-24T20:30:27.189341+00:00,8017033611084261128
1,E10002,Hamza Butt,M,1986-04-07,hamza.butt922@induswave.example,0347-4313277,Faisalabad Plant,cutting,Operator,G1,Contract,2016-02-15,,Active,14,2026-07-01 12:48:00,employees.csv,2026-10-24T20:30:27.189341+00:00,10211235729498233157
2,E10003,Saad Butt,M,1986-03-06,saad.butt122@induswave.example,0338-4383725,Korangi Plant,Maintenance,Senior Operator,G2,Permanent,2018-02-23,,Active,14,2026-07-01 17:47:00,employees.csv,2026-10-24T20:30:27.189341+00:00,8985980232758771058
3,E10004,Bilal Raza,M,23-Jul-1989,bilal.raza496@induswave.example,0317-1570993,SITE Plant,QUALITY CONTROL,Operator,G1,Permanent,2023-10-20,,Active,14,2026-07-01 15:18:00,employees.csv,2026-10-24T20:30:27.189341+00:00,9741957867171209411
4,E10005,Junaid Rana,M,11/01/1978,junaid.rana486@induswave.example,0313-7429960,Korangi Plant,Spinning,Operator,G1,Permanent,2020-07-25,,Active,14,2026-07-01 16:09:00,employees.csv,2026-10-24T20:30:27.189341+00:00,3821591432219566464


In [55]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query(
    "SELECT * FROM stg_employees",
    conn
)

columns_to_check = [
    "emp_id",
    "full_name",
    "department",
    "hire_date",
    "updated_at"
]

for column in columns_to_check:
    missing = (
        df[column].isna() | df[column].astype(str).str.strip().eq("")
    ).sum()

    print(f"{column}: {missing} missing values")

conn.close()

emp_id: 0 missing values
full_name: 0 missing values
department: 0 missing values
hire_date: 0 missing values
updated_at: 0 missing values


In [56]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT emp_id) AS unique_employees
    FROM stg_employees
""", conn)

display(result)

conn.close()

,total_rows,unique_employees
0,1266,1266


In [57]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query("""
    SELECT emp_id, hire_date, updated_at
    FROM stg_employees
    LIMIT 10
""", conn)

display(df)

conn.close()

,emp_id,hire_date,updated_at
0,E10001,19/04/2019,2026-06-02 12:25:00
1,E10002,2016-02-15,2026-07-01 12:48:00
2,E10003,2018-02-23,2026-07-01 17:47:00
3,E10004,2023-10-20,2026-07-01 15:18:00
4,E10005,2020-07-25,2026-07-01 16:09:00
5,E10006,11/05/2015,2026-07-01 16:08:00
6,E10007,2019-05-09,2026-07-01 13:32:00
7,E10008,2012-09-30,2026-08-31 13:53:00
8,E10009,2014-11-24,2026-07-01 12:04:00
9,E10010,17-Feb-2013,2026-07-01 11:45:00


In [58]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query("""
    SELECT hire_date
    FROM stg_employees
""", conn)

print("Total employees:", len(df))

print("\nHire date ke examples:")
print(df["hire_date"].value_counts().head(15))

conn.close()

Total employees: 1266

Hire date ke examples:
hire_date
2019-01-21     3
2013-03-24     3
30/08/2024     2
2025-07-05     2
2021-02-19     2
21-Jun-2020    2
18/01/2012     2
12/07/2014     2
2013-07-26     2
2023-05-22     2
2019-12-25     2
2026-03-10     2
24/06/2021     2
05/08/2014     2
2026-09-15     2
Name: count, dtype: int64


In [59]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query(
    "SELECT hire_date FROM stg_employees", conn
)

def identify_format(date):
    date = str(date).strip()

    if "/" in date:
        return "Slash format (DD/MM/YYYY)"
    elif "-" in date and len(date) == 10 and date[4] == "-":
        return "ISO format (YYYY-MM-DD)"
    elif "-" in date:
        return "Text month format (e.g. 21-Jun-2020)"
    else:
        return "Other format"

df["date_format"] = df["hire_date"].apply(identify_format)

print(df["date_format"].value_counts())

conn.close()

date_format
ISO format (YYYY-MM-DD)                 762
Slash format (DD/MM/YYYY)               312
Text month format (e.g. 21-Jun-2020)    192
Name: count, dtype: int64


In [60]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query(
    "SELECT * FROM stg_employees", conn
)

df["hire_date_clean"] = pd.to_datetime(
    df["hire_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
).dt.strftime("%Y-%m-%d")

print("Total employees:", len(df))
print("Dates converted:", df["hire_date_clean"].notna().sum())
print("Dates not converted:", df["hire_date_clean"].isna().sum())

display(df[["emp_id", "hire_date", "hire_date_clean"]].head(10))

conn.close()

Total employees: 1266
Dates converted: 1266
Dates not converted: 0


,emp_id,hire_date,hire_date_clean
0,E10001,19/04/2019,2019-04-19
1,E10002,2016-02-15,2016-02-15
2,E10003,2018-02-23,2018-02-23
3,E10004,2023-10-20,2023-10-20
4,E10005,2020-07-25,2020-07-25
5,E10006,11/05/2015,2015-05-11
6,E10007,2019-05-09,2019-09-05
7,E10008,2012-09-30,2012-09-30
8,E10009,2014-11-24,2014-11-24
9,E10010,17-Feb-2013,2013-02-17


In [61]:
import pandas as pd

def clean_hire_date(value):
    value = str(value).strip()

    # ISO format: YYYY-MM-DD
    if len(value) >= 10 and value[4] == "-" and value[7] == "-":
        date = pd.to_datetime(value, format="%Y-%m-%d", errors="coerce")

    # Slash format: DD/MM/YYYY
    elif "/" in value:
        date = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")

    # Text month format: DD-Mon-YYYY
    else:
        date = pd.to_datetime(value, format="%d-%b-%Y", errors="coerce")

    return date.strftime("%Y-%m-%d") if pd.notna(date) else None


df["hire_date_clean"] = df["hire_date"].apply(clean_hire_date)

print("Total employees:", len(df))
print("Dates converted:", df["hire_date_clean"].notna().sum())
print("Dates not converted:", df["hire_date_clean"].isna().sum())

display(df[["emp_id", "hire_date", "hire_date_clean"]].head(10))

Total employees: 1266
Dates converted: 1266
Dates not converted: 0


,emp_id,hire_date,hire_date_clean
0,E10001,19/04/2019,2019-04-19
1,E10002,2016-02-15,2016-02-15
2,E10003,2018-02-23,2018-02-23
3,E10004,2023-10-20,2023-10-20
4,E10005,2020-07-25,2020-07-25
5,E10006,11/05/2015,2015-05-11
6,E10007,2019-05-09,2019-05-09
7,E10008,2012-09-30,2012-09-30
8,E10009,2014-11-24,2014-11-24
9,E10010,17-Feb-2013,2013-02-17


In [62]:
import sqlite3

conn = sqlite3.connect("peopleops_hackathon.db")

df.to_sql(
    "clean_employees",
    conn,
    if_exists="replace",
    index=False
)

print("Cleaned employees table saved successfully!")

check = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT emp_id) AS unique_employees
    FROM clean_employees
""", conn)

display(check)

conn.close()

Cleaned employees table saved successfully!


,total_rows,unique_employees
0,1266,1266


In [63]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT emp_id, hire_date, hire_date_clean
    FROM clean_employees
    WHERE hire_date_clean > date('now')
""", conn)

print("Future hire dates:", len(result))
display(result.head(10))

conn.close()

Future hire dates: 0


,emp_id,hire_date,hire_date_clean


In [64]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT
        department,
        COUNT(*) AS employee_count
    FROM clean_employees
    GROUP BY department
    ORDER BY employee_count DESC
""", conn)

display(result)

conn.close()

,department,employee_count
0,Stitching,98
1,Weaving,77
2,Dyeing,73
3,Maintenance,70
4,Quality Control,69
...,...,...
75,ADMINISTRATION,1
76,Sales & Marketing,1
77,Production Planning,1
78,Procurement,1


In [65]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT DISTINCT department
    FROM clean_employees
    ORDER BY department
""", conn)

print("Total distinct department names:", len(result))
display(result)

conn.close()

Total distinct department names: 80


,department
0,Administration
1,Cutting
2,Dyeing
3,Finance
4,Finishing
...,...
75,stitching
76,supply chain
77,utilities
78,warehouse


In [66]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT *
    FROM raw_departments
    LIMIT 10
""", conn)

display(result)

conn.close()

,dept_id,dept_name,plant,cost_center,previous_name,renamed_on,_source_file,_loaded_at,_row_hash
0,D01,Human Resources,Head Office,CC-HO-01,,,departments.csv,2026-10-24T20:30:27.496378+00:00,14642499454455055522
1,D02,Finance,Head Office,CC-HO-02,,,departments.csv,2026-10-24T20:30:27.496378+00:00,8589762594317688205
2,D03,IT,Head Office,CC-HO-03,,,departments.csv,2026-10-24T20:30:27.496378+00:00,5193677743650524296
3,D04,Supply Chain,Head Office,CC-HO-04,Procurement,2026-07-01,departments.csv,2026-10-24T20:30:27.496378+00:00,3554395351112450257
4,D05,Sales & Marketing,Head Office,CC-HO-05,,,departments.csv,2026-10-24T20:30:27.496378+00:00,3351702163255188046
5,D06,Administration,Head Office,CC-HO-06,,,departments.csv,2026-10-24T20:30:27.496378+00:00,13821083176078003158
6,D07,Compliance,Head Office,CC-HO-07,,,departments.csv,2026-10-24T20:30:27.496378+00:00,5817069235243224534
7,D08,Spinning,Korangi Plant,CC-KOR-08,,,departments.csv,2026-10-24T20:30:27.496378+00:00,7914471134180199523
8,D09,Weaving,Korangi Plant,CC-KOR-09,,,departments.csv,2026-10-24T20:30:27.496378+00:00,8535202486988219868
9,D10,Maintenance,Korangi Plant,CC-KOR-10,,,departments.csv,2026-10-24T20:30:27.496378+00:00,195554735061985433


In [67]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT dept_id, dept_name, previous_name, renamed_on
    FROM raw_departments
    ORDER BY dept_id
""", conn)

print("Official department records:", len(result))
display(result)

conn.close()

Official department records: 25


,dept_id,dept_name,previous_name,renamed_on
0,D01,Human Resources,,
1,D02,Finance,,
2,D03,IT,,
3,D04,Supply Chain,Procurement,2026-07-01
4,D05,Sales & Marketing,,
5,D06,Administration,,
6,D07,Compliance,,
7,D08,Spinning,,
8,D09,Weaving,,
9,D10,Maintenance,,


In [68]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

employee_depts = pd.read_sql_query("""
    SELECT department, COUNT(*) AS employee_count
    FROM clean_employees
    GROUP BY department
    ORDER BY department
""", conn)

official_depts = pd.read_sql_query("""
    SELECT dept_name, previous_name, renamed_on
    FROM raw_departments
""", conn)

official_current = set(
    official_depts["dept_name"].dropna().str.strip().str.casefold()
)
official_previous = set(
    official_depts["previous_name"].dropna().str.strip().str.casefold()
)

def classify_department(name):
    cleaned = str(name).strip().casefold()

    if cleaned in official_current:
        return "Official current name"
    elif cleaned in official_previous:
        return "Official previous name"
    return "Needs investigation"

employee_depts["status"] = employee_depts["department"].apply(
    classify_department
)

display(employee_depts)

print("\nDepartment status summary:")
display(employee_depts.groupby("status")["employee_count"].sum())

conn.close()


,department,employee_count,status
0,Administration,3,Official current name
1,Cutting,10,Official current name
2,Dyeing,13,Official current name
3,Finance,2,Official current name
4,Finishing,14,Official current name
...,...,...,...
75,stitching,22,Official current name
76,supply chain,1,Official current name
77,utilities,5,Official current name
78,warehouse,9,Official current name



Department status summary:


status
Official current name     1223
Official previous name      43
Name: employee_count, dtype: int64

In [69]:

previous_matches = employee_depts[
    employee_depts["status"] == "Official previous name"
]

display(previous_matches)
print("Total employees using previous department names:",
      previous_matches["employee_count"].sum())


,department,employee_count,status
8,Processing,5,Official previous name
9,Procurement,1,Official previous name
36,PROCESSING,8,Official previous name
37,PROCUREMENT,1,Official previous name
40,Processing,16,Official previous name
41,Procurement,6,Official previous name
69,processing,3,Official previous name
70,procurement,3,Official previous name


Total employees using previous department names: 43


In [70]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query("SELECT * FROM clean_employees", conn)

def standardize_department(value):
    name = str(value).strip()
    key = name.casefold()

    if key == "processing":
        return "Finishing"
    elif key == "procurement":
        return "Supply Chain"
    else:
        # Official capitalization ko match karne ke liye
        official_names = {
            "administration": "Administration",
            "cutting": "Cutting",
            "dyeing": "Dyeing",
            "finance": "Finance",
            "finishing": "Finishing",
            "it": "IT",
            "stitching": "Stitching",
            "supply chain": "Supply Chain",
            "utilities": "Utilities",
            "warehouse": "Warehouse",
            "weaving": "Weaving",
        }
        return official_names.get(key, name)

df["department_clean"] = df["department"].apply(standardize_department)

df.to_sql("stg_employees_departments", conn,
          if_exists="replace", index=False)

print("Rows:", len(df))
print("Unique employee IDs:", df["emp_id"].nunique())
print("Departments after standardization:",
      df["department_clean"].nunique())

display(
    df.groupby("department_clean")
      .size()
      .reset_index(name="employee_count")
      .sort_values("department_clean")
)

conn.close()


Rows: 1266
Unique employee IDs: 1266
Departments after standardization: 35


,department_clean,employee_count
0,Administration,19
1,COMPLIANCE,1
2,Compliance,11
3,Cutting,83
4,Dyeing,115
5,Finance,20
6,Finishing,102
7,HUMAN RESOURCES,3
8,Human Resources,11
9,IT,15


In [71]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

df = pd.read_sql_query(
    "SELECT * FROM clean_employees", conn
)

official = pd.read_sql_query(
    "SELECT dept_name, previous_name FROM raw_departments",
    conn
)

# Case-insensitive mapping: current names and previous names
name_map = {}

for _, row in official.iterrows():
    current = str(row["dept_name"]).strip()
    name_map[current.casefold()] = current

    previous = row["previous_name"]
    if pd.notna(previous) and str(previous).strip():
        name_map[str(previous).strip().casefold()] = current

df["department_clean"] = df["department"].apply(
    lambda x: name_map.get(str(x).strip().casefold(), str(x).strip())
)

df.to_sql(
    "stg_employees_departments",
    conn,
    if_exists="replace",
    index=False
)

print("Rows:", len(df))
print("Unique employee IDs:", df["emp_id"].nunique())
print("Standardized department values:",
      df["department_clean"].nunique())

display(
    df.groupby("department_clean")
      .size()
      .reset_index(name="employee_count")
      .sort_values("department_clean")
)

conn.close()


Rows: 1266
Unique employee IDs: 1266
Standardized department values: 19


,department_clean,employee_count
0,Administration,19
1,Compliance,13
2,Cutting,83
3,Dyeing,115
4,Finance,20
5,Finishing,102
6,Human Resources,16
7,IT,15
8,Maintenance,109
9,Packing,65


In [72]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT DISTINCT e.department_clean
    FROM stg_employees_departments e
    LEFT JOIN raw_departments d
      ON LOWER(TRIM(e.department_clean)) =
         LOWER(TRIM(d.dept_name))
    WHERE d.dept_name IS NULL
    ORDER BY e.department_clean
""", conn)

print("Unmatched department names:", len(result))
display(result)

conn.close()


Unmatched department names: 0


,department_clean


In [73]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT emp_id, COUNT(*) AS record_count
    FROM stg_employees_departments
    GROUP BY emp_id
    HAVING COUNT(*) > 1
    ORDER BY record_count DESC
""", conn)

print("Duplicate employee IDs:", len(result))
display(result.head(10))

conn.close()


Duplicate employee IDs: 0


,emp_id,record_count


In [74]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT
        SUM(CASE WHEN emp_id IS NULL OR TRIM(emp_id) = ''
            THEN 1 ELSE 0 END) AS missing_emp_id,
        SUM(CASE WHEN full_name IS NULL OR TRIM(full_name) = ''
            THEN 1 ELSE 0 END) AS missing_full_name,
        SUM(CASE WHEN department_clean IS NULL
                      OR TRIM(department_clean) = ''
            THEN 1 ELSE 0 END) AS missing_department,
        SUM(CASE WHEN hire_date_clean IS NULL
                      OR TRIM(hire_date_clean) = ''
            THEN 1 ELSE 0 END) AS missing_hire_date
    FROM stg_employees_departments
""", conn)

display(result)

conn.close()


,missing_emp_id,missing_full_name,missing_department,missing_hire_date
0,0,0,0,0


In [75]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT emp_id, full_name, hire_date_clean
    FROM stg_employees_departments
    WHERE DATE(hire_date_clean) > DATE('now')
    ORDER BY hire_date_clean
""", conn)

print("Employees with future hire dates:", len(result))
display(result.head(20))

conn.close()


Employees with future hire dates: 0


,emp_id,full_name,hire_date_clean


In [76]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

columns = pd.read_sql_query("""
    PRAGMA table_info(stg_employees_departments)
""", conn)

display(columns[["name", "type"]])

conn.close()


,name,type
0,emp_id,TEXT
1,full_name,TEXT
2,gender,TEXT
3,date_of_birth,TEXT
4,email,TEXT
5,phone,TEXT
6,plant,TEXT
7,department,TEXT
8,designation,TEXT
9,grade,TEXT


In [77]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT
        status,
        COUNT(*) AS employee_count
    FROM stg_employees_departments
    GROUP BY status
    ORDER BY employee_count DESC
""", conn)

display(result)

conn.close()


,status,employee_count
0,Active,1176
1,Separated,90


In [78]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT
        status,
        COUNT(*) AS total_employees,
        SUM(
            CASE
                WHEN separation_date IS NOT NULL
                     AND TRIM(separation_date) <> ''
                THEN 1 ELSE 0
            END
        ) AS with_separation_date,
        SUM(
            CASE
                WHEN separation_date IS NULL
                     OR TRIM(separation_date) = ''
                THEN 1 ELSE 0
            END
        ) AS without_separation_date
    FROM stg_employees_departments
    GROUP BY status
    ORDER BY status
""", conn)

display(result)

conn.close()


,status,total_employees,with_separation_date,without_separation_date
0,Active,1176,0,1176
1,Separated,90,90,0


In [79]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

result = pd.read_sql_query("""
    SELECT emp_id, full_name, hire_date_clean, separation_date
    FROM stg_employees_departments
    WHERE LOWER(TRIM(status)) = 'separated'
      AND DATE(separation_date) < DATE(hire_date_clean)
    ORDER BY emp_id
""", conn)

print("Separation dates before hire dates:", len(result))
display(result)

conn.close()


Separation dates before hire dates: 0


,emp_id,full_name,hire_date_clean,separation_date


In [80]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

for table in [
    "raw_biometric_punches",
    "raw_device_mapping",
    "raw_shift_roster",
    "raw_holidays"
]:
    print(f"\n--- {table} ---")
    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})", conn
    )
    print(columns["name"].tolist())

conn.close()



--- raw_biometric_punches ---
['punch_id', 'device_id', 'device_user_id', 'punch_time', '_source_file', '_loaded_at', '_row_hash']

--- raw_device_mapping ---
['device_id', 'device_user_id', 'emp_id', 'enrolled_on', '_source_file', '_loaded_at', '_row_hash']

--- raw_shift_roster ---
['emp_id', 'week_start', 'shift_code', '_source_file', '_loaded_at', '_row_hash']

--- raw_holidays ---
['holiday_date', 'holiday_name', 'scope', '_source_file', '_loaded_at', '_row_hash']


In [81]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

tables = [
    "raw_biometric_punches",
    "raw_device_mapping",
    "raw_shift_roster",
    "raw_holidays"
]

for table in tables:
    print(f"\n--- {table} ---")
    df_sample = pd.read_sql_query(
        f"SELECT * FROM {table} LIMIT 5",
        conn
    )
    display(df_sample)

conn.close()



--- raw_biometric_punches ---


,punch_id,device_id,device_user_id,punch_time,_source_file,_loaded_at,_row_hash
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,13151404056921306881
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,4785220893890696708
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,14167321896777283984
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,1786935183534052787
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,17863016332569932636



--- raw_device_mapping ---


,device_id,device_user_id,emp_id,enrolled_on,_source_file,_loaded_at,_row_hash
0,DEV-SITE-01,16779,E10250,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,14021375231114033208
1,DEV-FSD-01,10465,E10591,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,4870097045872590770
2,DEV-KOR-01,36101,E10916,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,2037419434898722211
3,DEV-FSD-01,7339,E10865,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,941764935120448705
4,DEV-KOR-01,26390,E10683,2025-01-31,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,4786636877311686356



--- raw_shift_roster ---


,emp_id,week_start,shift_code,_source_file,_loaded_at,_row_hash
0,E10796,2026-09-28,NIGHT,shift_roster.csv,2026-10-24T20:30:27.613401+00:00,3613937119620332530
1,E11106,2026-04-20,NIGHT,shift_roster.csv,2026-10-24T20:30:27.613401+00:00,5040001608662463125
2,E11119,2026-10-12,NIGHT,shift_roster.csv,2026-10-24T20:30:27.613401+00:00,3262971126381068323
3,E10370,2026-06-29,DAY,shift_roster.csv,2026-10-24T20:30:27.613401+00:00,3367276027572626633
4,E10735,2026-04-20,EVE,shift_roster.csv,2026-10-24T20:30:27.613401+00:00,13398900503061615823



--- raw_holidays ---


,holiday_date,holiday_name,scope,_source_file,_loaded_at,_row_hash
0,2026-04-25,Local holiday,FSD,holidays.csv,2026-10-24T20:30:32.718442+00:00,13136193164358180783
1,2026-05-01,Labour Day,ALL,holidays.csv,2026-10-24T20:30:32.718442+00:00,914514805966857166
2,2026-05-27,Eid ul Adha,ALL,holidays.csv,2026-10-24T20:30:32.718442+00:00,1257193737384185842
3,2026-05-28,Eid ul Adha,ALL,holidays.csv,2026-10-24T20:30:32.718442+00:00,4637140462422946651
4,2026-05-29,Eid ul Adha,ALL,holidays.csv,2026-10-24T20:30:32.718442+00:00,11964516763642058089


In [82]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

for table in ["raw_biometric_punches", "raw_device_mapping"]:
    print(f"\n--- {table} ---")
    display(pd.read_sql_query(
        f"SELECT * FROM {table} LIMIT 5", conn
    ))

conn.close()



--- raw_biometric_punches ---


,punch_id,device_id,device_user_id,punch_time,_source_file,_loaded_at,_row_hash
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,13151404056921306881
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,4785220893890696708
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,14167321896777283984
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,1786935183534052787
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33,biometric_punches.csv,2026-10-24T20:30:29.058266+00:00,17863016332569932636



--- raw_device_mapping ---


,device_id,device_user_id,emp_id,enrolled_on,_source_file,_loaded_at,_row_hash
0,DEV-SITE-01,16779,E10250,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,14021375231114033208
1,DEV-FSD-01,10465,E10591,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,4870097045872590770
2,DEV-KOR-01,36101,E10916,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,2037419434898722211
3,DEV-FSD-01,7339,E10865,2024-01-01,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,941764935120448705
4,DEV-KOR-01,26390,E10683,2025-01-31,device_mapping.csv,2026-10-24T20:30:28.029820+00:00,4786636877311686356


In [83]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Total records aur unique device-user combinations
mapping_check = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT device_id || '-' || device_user_id)
        AS unique_device_users,
    COUNT(DISTINCT emp_id) AS unique_employees
FROM raw_device_mapping;
""", conn)

display(mapping_check)

# Duplicate device-user combinations
duplicates = pd.read_sql_query("""
SELECT
    device_id,
    device_user_id,
    COUNT(*) AS record_count
FROM raw_device_mapping
GROUP BY device_id, device_user_id
HAVING COUNT(*) > 1
LIMIT 10;
""", conn)

print("Duplicate device mappings:")
display(duplicates)

conn.close()

,total_rows,unique_device_users,unique_employees
0,1260,1260,1260


Duplicate device mappings:


,device_id,device_user_id,record_count


In [84]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

query = """
SELECT
    p.punch_id,
    p.device_id,
    p.device_user_id,
    p.punch_time,
    m.emp_id,
    e.full_name,
    e.department_clean
FROM raw_biometric_punches AS p
LEFT JOIN raw_device_mapping AS m
    ON p.device_id = m.device_id
   AND p.device_user_id = m.device_user_id
LEFT JOIN stg_employees_departments AS e
    ON m.emp_id = e.emp_id
"""

attendance_base = pd.read_sql_query(query, conn)

print("Total biometric punches:", len(attendance_base))
print("Punches with employee mapping:", attendance_base["emp_id"].notna().sum())
print("Punches without employee mapping:", attendance_base["emp_id"].isna().sum())

display(attendance_base.head(10))


Total biometric punches: 345280
Punches with employee mapping: 338318
Punches without employee mapping: 6962


,punch_id,device_id,device_user_id,punch_time,emp_id,full_name,department_clean
0,P00000001,DEV-KOR-01,84982,2026-04-01 05:35:12,E10543,Yasir Mirza,Spinning
1,P00000002,DEV-SITE-01,28825,2026-04-01 05:35:14,E10568,Sohail Khan,Quality Control
2,P00000003,DEV-KOR-01,78716,2026-04-01 05:35:30,E10384,Saad Awan,Maintenance
3,P00000004,DEV-FSD-01,88557,2026-04-01 05:35:33,E11112,Iqra Raza,Cutting
4,P00000005,DEV-SITE-01,92891,2026-04-01 05:35:33,E10880,Hina Memon,Utilities
5,P00000006,DEV-FSD-01,82007,2026-04-01 05:35:35,E10897,Rabia Hussain,Stitching
6,P00000007,DEV-SITE-01,20901,2026-04-01 05:35:47,E11235,Haris Chaudhry,Finishing
7,P00000008,DEV-FSD-01,85626,2026-04-01 05:36:01,E11024,Adnan Awan,Cutting
8,P00000009,DEV-FSD-01,85888,2026-04-01 05:36:21,E10133,Bilal Khan,Cutting
9,P00000010,DEV-FSD-01,13660,2026-04-01 05:36:21,E10314,Kashif Baloch,Stitching


In [85]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Mapped punches: employee mapping available
mapped = attendance_base[attendance_base["emp_id"].notna()].copy()

# Unmatched punches: employee mapping unavailable
unmatched = attendance_base[attendance_base["emp_id"].isna()].copy()

# Save staging tables; raw tables remain unchanged
mapped.to_sql("stg_biometric_punches", conn, if_exists="replace", index=False)
unmatched.to_sql("dq_unmatched_punches", conn, if_exists="replace", index=False)

# Inspect shift roster and shift JSON structure
print("Shift roster summary:")
display(pd.read_sql_query("""
SELECT shift_code, COUNT(*) AS roster_rows,
       MIN(week_start) AS first_week,
       MAX(week_start) AS last_week
FROM raw_shift_roster
GROUP BY shift_code
ORDER BY shift_code;
""", conn))

print("JSON table columns:")
display(pd.read_sql_query("PRAGMA table_info(raw_json_files);", conn))

print("Saved staging counts:")
display(pd.read_sql_query("""
SELECT 'stg_biometric_punches' AS table_name, COUNT(*) AS rows
FROM stg_biometric_punches
UNION ALL
SELECT 'dq_unmatched_punches', COUNT(*)
FROM dq_unmatched_punches;
""", conn))

conn.close()


Shift roster summary:


,shift_code,roster_rows,first_week,last_week
0,DAY,10596,2026-03-30,2026-10-26
1,EVE,10606,2026-03-30,2026-10-26
2,GEN,4608,2026-03-30,2026-10-26
3,NIGHT,10592,2026-03-30,2026-10-26


JSON table columns:


,cid,name,type,notnull,dflt_value,pk
0,0,source_file,TEXT,0,None,0
1,1,raw_content,TEXT,0,None,0
2,2,loaded_at,TEXT,0,None,0


Saved staging counts:


,table_name,rows
0,stg_biometric_punches,338318
1,dq_unmatched_punches,6962


In [86]:

import sqlite3
import pandas as pd
import json

conn = sqlite3.connect("peopleops_hackathon.db")

# shifts.json ka content read karo
shift_file = pd.read_sql_query("""
SELECT source_file, raw_content
FROM raw_json_files
WHERE LOWER(source_file) LIKE '%shifts.json%'
LIMIT 1;
""", conn)

if shift_file.empty:
    print("shifts.json nahi mili. Available JSON files:")
    display(pd.read_sql_query(
        "SELECT source_file FROM raw_json_files;",
        conn
    ))
else:
    content = shift_file.loc[0, "raw_content"]
    try:
        parsed = json.loads(content)
        print(json.dumps(parsed, indent=2)[:8000])
    except (json.JSONDecodeError, TypeError):
        print(str(content)[:8000])

conn.close()


{
  "shifts": [
    {
      "shift_code": "DAY",
      "name": "Day",
      "start_time": "06:00",
      "end_time": "14:00",
      "crosses_midnight": false,
      "grace_minutes": 10,
      "applies_to": "Plant shift workers"
    },
    {
      "shift_code": "EVE",
      "name": "Evening",
      "start_time": "14:00",
      "end_time": "22:00",
      "crosses_midnight": false,
      "grace_minutes": 10,
      "applies_to": "Plant shift workers"
    },
    {
      "shift_code": "NIGHT",
      "name": "Night",
      "start_time": "22:00",
      "end_time": "06:00",
      "crosses_midnight": true,
      "grace_minutes": 10,
      "applies_to": "Plant shift workers"
    },
    {
      "shift_code": "GEN",
      "name": "General",
      "start_time": "09:00",
      "end_time": "17:30",
      "crosses_midnight": false,
      "grace_minutes": 10,
      "applies_to": "Head office and plant staff"
    }
  ]
}


In [87]:

import sqlite3
import pandas as pd
import json
from datetime import datetime, timedelta

conn = sqlite3.connect("peopleops_hackathon.db")

# Shift definitions source JSON se lo
shift_row = pd.read_sql_query("""
SELECT raw_content
FROM raw_json_files
WHERE LOWER(source_file) LIKE '%shifts.json%'
LIMIT 1
""", conn)

shift_data = json.loads(shift_row.loc[0, "raw_content"])
shifts = {
    s["shift_code"]: s
    for s in shift_data["shifts"]
}

# Weekly roster load karo
roster = pd.read_sql_query("""
SELECT emp_id, week_start, shift_code
FROM raw_shift_roster
""", conn)

roster["week_start"] = pd.to_datetime(roster["week_start"])

# Har weekly roster ko 7 daily scheduled shifts mein expand karo
daily_rows = []

for _, row in roster.iterrows():
    shift = shifts.get(row["shift_code"])
    if not shift:
        continue

    for day_offset in range(7):
        work_date = row["week_start"] + pd.Timedelta(days=day_offset)
        start_dt = pd.to_datetime(
            f"{work_date.date()} {shift['start_time']}"
        )
        end_dt = pd.to_datetime(
            f"{work_date.date()} {shift['end_time']}"
        )

        if shift["crosses_midnight"]:
            end_dt += pd.Timedelta(days=1)

        daily_rows.append({
            "emp_id": row["emp_id"],
            "attendance_date": str(work_date.date()),
            "shift_code": row["shift_code"],
            "shift_start": str(start_dt),
            "shift_end": str(end_dt),
            "grace_minutes": shift["grace_minutes"]
        })

daily_roster = pd.DataFrame(daily_rows)

# Save daily schedule; raw tables remain unchanged
daily_roster.to_sql(
    "stg_daily_roster",
    conn,
    if_exists="replace",
    index=False
)

print("Daily scheduled shifts:", len(daily_roster))
print("Unique employees:", daily_roster["emp_id"].nunique
      
print("Shift counts:")
display(daily_roster["shift_code"].value_counts().rename_axis(
    "shift_code"
).reset_index(name="daily_rows"))

print("Sample daily schedule:")
display(daily_roster.head(10))

conn.close()


Daily scheduled shifts: 254814
Unique employees: 1266
Shift counts:


,shift_code,daily_rows
0,EVE,74242
1,DAY,74172
2,NIGHT,74144
3,GEN,32256


Sample daily schedule:


,emp_id,attendance_date,shift_code,shift_start,shift_end,grace_minutes
0,E10796,2026-09-28,NIGHT,2026-09-28 22:00:00,2026-09-29 06:00:00,10
1,E10796,2026-09-29,NIGHT,2026-09-29 22:00:00,2026-09-30 06:00:00,10
2,E10796,2026-09-30,NIGHT,2026-09-30 22:00:00,2026-10-01 06:00:00,10
3,E10796,2026-10-01,NIGHT,2026-10-01 22:00:00,2026-10-02 06:00:00,10
4,E10796,2026-10-02,NIGHT,2026-10-02 22:00:00,2026-10-03 06:00:00,10
5,E10796,2026-10-03,NIGHT,2026-10-03 22:00:00,2026-10-04 06:00:00,10
6,E10796,2026-10-04,NIGHT,2026-10-04 22:00:00,2026-10-05 06:00:00,10
7,E11106,2026-04-20,NIGHT,2026-04-20 22:00:00,2026-04-21 06:00:00,10
8,E11106,2026-04-21,NIGHT,2026-04-21 22:00:00,2026-04-22 06:00:00,10
9,E11106,2026-04-22,NIGHT,2026-04-22 22:00:00,2026-04-23 06:00:00,10


In [88]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Query ko tez rakhne ke liye indexes
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_punch_emp_time
ON stg_biometric_punches(emp_id, punch_time)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_roster_emp_date
ON stg_daily_roster(emp_id, attendance_date)
""")

# Har shift ke liye punches match karo
conn.execute("DROP TABLE IF EXISTS stg_shift_punch_summary")

conn.execute("""
CREATE TABLE stg_shift_punch_summary AS
SELECT
    r.emp_id,
    r.attendance_date,
    r.shift_code,
    r.shift_start,
    r.shift_end,
    r.grace_minutes,
    MIN(p.punch_time) AS actual_in,
    MAX(p.punch_time) AS actual_out,
    COUNT(p.punch_id) AS punch_count
FROM stg_daily_roster r
LEFT JOIN stg_biometric_punches p
    ON p.emp_id = r.emp_id
   AND date(p.punch_time) BETWEEN
       r.attendance_date AND date(r.attendance_date, '+1 day')
   AND p.punch_time BETWEEN
       datetime(r.shift_start, '-2 hours')
       AND datetime(r.shift_end, '+2 hours')
GROUP BY
    r.emp_id,
    r.attendance_date,
    r.shift_code,
    r.shift_start,
    r.shift_end,
    r.grace_minutes
""")

print("Shift punch summary:")
display(pd.read_sql_query("""
SELECT
    COUNT(*) AS scheduled_shifts,
    SUM(CASE WHEN punch_count = 0 THEN 1 ELSE 0 END)
        AS shifts_without_punches,
    SUM(CASE WHEN punch_count = 1 THEN 1 ELSE 0 END)
        AS shifts_with_one_punch,
    SUM(CASE WHEN punch_count >= 2 THEN 1 ELSE 0 END)
        AS shifts_with_multiple_punches
FROM stg_shift_punch_summary
""", conn))

print("Sample matched shifts:")
display(pd.read_sql_query("""
SELECT *
FROM stg_shift_punch_summary
WHERE punch_count > 0
LIMIT 10
""", conn))

conn.close()


Shift punch summary:


,scheduled_shifts,shifts_without_punches,shifts_with_one_punch,shifts_with_multiple_punches
0,254814,86497,17951,150366


Sample matched shifts:


,emp_id,attendance_date,shift_code,shift_start,shift_end,grace_minutes,actual_in,actual_out,punch_count
0,E10001,2026-04-01,DAY,2026-04-01 06:00:00,2026-04-01 14:00:00,10,2026-04-01 05:46:03,2026-04-01 14:07:09,3
1,E10001,2026-04-02,DAY,2026-04-02 06:00:00,2026-04-02 14:00:00,10,2026-04-02 05:48:47,2026-04-02 14:22:29,2
2,E10001,2026-04-03,DAY,2026-04-03 06:00:00,2026-04-03 14:00:00,10,2026-04-03 05:53:32,2026-04-03 05:53:32,1
3,E10001,2026-04-04,DAY,2026-04-04 06:00:00,2026-04-04 14:00:00,10,2026-04-04 05:50:38,2026-04-04 14:22:37,2
4,E10001,2026-04-06,EVE,2026-04-06 14:00:00,2026-04-06 22:00:00,10,2026-04-06 13:54:27,2026-04-06 22:13:15,2
5,E10001,2026-04-07,EVE,2026-04-07 14:00:00,2026-04-07 22:00:00,10,2026-04-07 14:09:13,2026-04-07 22:16:02,2
6,E10001,2026-04-09,EVE,2026-04-09 14:00:00,2026-04-09 22:00:00,10,2026-04-09 13:48:52,2026-04-09 22:06:46,2
7,E10001,2026-04-10,EVE,2026-04-10 14:00:00,2026-04-10 22:00:00,10,2026-04-10 13:36:55,2026-04-10 22:11:58,2
8,E10001,2026-04-11,EVE,2026-04-11 14:00:00,2026-04-11 22:00:00,10,2026-04-11 15:08:03,2026-04-11 22:03:21,2
9,E10001,2026-04-13,NIGHT,2026-04-13 22:00:00,2026-04-14 06:00:00,10,2026-04-13 21:37:47,2026-04-14 06:12:35,2


In [89]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Har punch ko nearest eligible shift assign karo
conn.execute("DROP TABLE IF EXISTS stg_punch_shift_assignment")

conn.execute("""
CREATE TABLE stg_punch_shift_assignment AS
WITH candidates AS (
    SELECT
        p.punch_id,
        p.punch_time,
        r.emp_id,
        r.attendance_date,
        r.shift_code,
        r.shift_start,
        r.shift_end,
        r.grace_minutes,
        CASE
            WHEN p.punch_time < r.shift_start
                THEN (julianday(r.shift_start)
                      - julianday(p.punch_time)) * 86400
            WHEN p.punch_time > r.shift_end
                THEN (julianday(p.punch_time)
                      - julianday(r.shift_end)) * 86400
            ELSE 0
        END AS distance_seconds
    FROM stg_biometric_punches p
    JOIN stg_daily_roster r
      ON p.emp_id = r.emp_id
     AND p.punch_time BETWEEN
         datetime(r.shift_start, '-2 hours')
         AND datetime(r.shift_end, '+2 hours')
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY punch_id
            ORDER BY distance_seconds, attendance_date
        ) AS rn
    FROM candidates
)
SELECT
    punch_id, punch_time, emp_id, attendance_date,
    shift_code, shift_start, shift_end, grace_minutes
FROM ranked
WHERE rn = 1
""")

# Assigned punches se shift summary dobara banao
conn.execute("DROP TABLE IF EXISTS stg_shift_punch_summary")

conn.execute("""
CREATE TABLE stg_shift_punch_summary AS
SELECT
    r.emp_id,
    r.attendance_date,
    r.shift_code,
    r.shift_start,
    r.shift_end,
    r.grace_minutes,
    MIN(a.punch_time) AS actual_in,
    MAX(a.punch_time) AS actual_out,
    COUNT(a.punch_id) AS punch_count
FROM stg_daily_roster r
LEFT JOIN stg_punch_shift_assignment a
  ON a.emp_id = r.emp_id
 AND a.attendance_date = r.attendance_date
 AND a.shift_code = r.shift_code
GROUP BY
    r.emp_id, r.attendance_date, r.shift_code,
    r.shift_start, r.shift_end, r.grace_minutes
""")

print("Punch assignment coverage:")
display(pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM raw_biometric_punches) AS all_raw_punches,
    (SELECT COUNT(*) FROM stg_biometric_punches) AS mapped_punches,
    (SELECT COUNT(*) FROM stg_punch_shift_assignment) AS assigned_punches
""", conn))

print("Revised shift summary:")
display(pd.read_sql_query("""
SELECT
    COUNT(*) AS scheduled_shifts,
    SUM(CASE WHEN punch_count = 0 THEN 1 ELSE 0 END)
        AS shifts_without_punches,
    SUM(CASE WHEN punch_count = 1 THEN 1 ELSE 0 END)
        AS shifts_with_one_punch,
    SUM(CASE WHEN punch_count >= 2 THEN 1 ELSE 0 END)
        AS shifts_with_multiple_punches
FROM stg_shift_punch_summary
""", conn))

conn.close()


Punch assignment coverage:


,all_raw_punches,mapped_punches,assigned_punches
0,345280,338318,325243


Revised shift summary:


,scheduled_shifts,shifts_without_punches,shifts_with_one_punch,shifts_with_multiple_punches
0,254814,90376,18261,146177


In [90]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("Unassigned punches by date:")
display(pd.read_sql_query("""
SELECT
    date(p.punch_time) AS punch_date,
    COUNT(*) AS unassigned_count
FROM stg_biometric_punches p
LEFT JOIN stg_punch_shift_assignment a
    ON p.punch_id = a.punch_id
WHERE a.punch_id IS NULL
GROUP BY date(p.punch_time)
ORDER BY punch_date
LIMIT 15
""", conn))

print("Punches outside roster date range:")
display(pd.read_sql_query("""
SELECT
    MIN(date(p.punch_time)) AS first_unassigned_date,
    MAX(date(p.punch_time)) AS last_unassigned_date,
    COUNT(*) AS unassigned_punches
FROM stg_biometric_punches p
LEFT JOIN stg_punch_shift_assignment a
    ON p.punch_id = a.punch_id
WHERE a.punch_id IS NULL
""", conn))

conn.close()


Unassigned punches by date:


,punch_date,unassigned_count
0,2026-04-01,27
1,2026-04-02,47
2,2026-04-03,44
3,2026-04-04,44
4,2026-04-05,26
5,2026-04-06,58
6,2026-04-07,96
7,2026-04-08,83
8,2026-04-09,86
9,2026-04-10,76


Punches outside roster date range:


,first_unassigned_date,last_unassigned_date,unassigned_punches
0,2026-04-01,2026-10-01,13075


In [91]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

query = """
SELECT
    CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM stg_daily_roster r
            WHERE r.emp_id = p.emp_id
              AND p.punch_time BETWEEN
                  datetime(r.shift_start, '-2 hours')
                  AND datetime(r.shift_end, '+2 hours')
        )
        THEN 'No eligible shift window'
        ELSE 'Eligible window exists but not assigned'
    END AS reason,
    COUNT(*) AS punch_count
FROM stg_biometric_punches p
LEFT JOIN stg_punch_shift_assignment a
    ON p.punch_id = a.punch_id
WHERE a.punch_id IS NULL
GROUP BY reason
"""

display(pd.read_sql_query(query, conn))

print("Holiday records:")
display(pd.read_sql_query("""
SELECT scope, COUNT(*) AS holiday_count,
       MIN(holiday_date) AS first_date,
       MAX(holiday_date) AS last_date
FROM raw_holidays
GROUP BY scope
""", conn))

print("Leave JSON files:")
display(pd.read_sql_query("""
SELECT source_file, length(raw_content) AS content_length
FROM raw_json_files
""", conn))

conn.close()


,reason,punch_count
0,No eligible shift window,13075


Holiday records:


,scope,holiday_count,first_date,last_date
0,ALL,8,2026-05-01,2026-08-26
1,FSD,2,2026-04-25,2026-07-18
2,KOR,1,2026-09-12,2026-09-12
3,SITE,1,2026-06-13,2026-06-13


Leave JSON files:


,source_file,content_length
0,leave_requests.json,3133899
1,shifts.json,915


In [92]:

import sqlite3
import pandas as pd
import json

conn = sqlite3.connect("peopleops_hackathon.db")

row = pd.read_sql_query("""
SELECT raw_content
FROM raw_json_files
WHERE source_file = 'leave_requests.json'
LIMIT 1
""", conn)

if row.empty:
    print("Leave requests file database mein nahi mili.")
else:
    leave_data = json.loads(row.loc[0, "raw_content"])

    print("Data type:", type(leave_data).__name__)

    if isinstance(leave_data, list):
        print("Total leave records:", len(leave_data))
        display(pd.json_normalize(leave_data).head(5))
    elif isinstance(leave_data, dict):
        print("Top-level keys:", list(leave_data.keys()))
        for key, value in leave_data.items():
            print(f"\nKey: {key} | Type: {type(value).__name__}")
            if isinstance(value, list):
                print("Records:", len(value))
                display(pd.json_normalize(value).head(5))
            elif isinstance(value, dict):
                print("Sample keys:", list(value.keys())[:20])
            else:
                print(str(value)[:500])

conn.close()


Data type: list
Total leave records: 5372


,request_id,submitted_at,approvals,cancelled_at,employee.emp_id,employee.plant,leave.type,leave.from,leave.to,leave.days_requested
0,LR000001,2026-04-04T10:10:00,"[{'step': 1, 'approver_role': 'Manager', 'appr...",2026-04-05T21:10:00,E10001,Korangi Plant,Casual,2026-04-08,2026-04-08,1
1,LR000002,2026-04-14T14:18:00,"[{'step': 1, 'approver_role': 'Manager', 'appr...",2026-04-16T09:18:00,E10001,Korangi Plant,Casual,2026-04-22,2026-04-22,1
2,LR000003,2026-05-07T10:27:00,"[{'step': 1, 'approver_role': 'Manager', 'appr...",NaN,E10001,Korangi Plant,Annual,2026-05-13,2026-05-14,2
3,LR000004,2026-04-16T10:27:00,"[{'step': 1, 'approver_role': 'Manager', 'appr...",NaN,E10001,Korangi Plant,Annual,2026-04-21,2026-04-21,1
4,LR000005,2026-05-31T11:20:00,"[{'step': 1, 'approver_role': 'Manager', 'appr...",NaN,E10003,Korangi Plant,Annual,2026-06-04,2026-06-04,1


In [95]:
import sqlite3
import json
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

raw = pd.read_sql_query("""
SELECT raw_content
FROM raw_json_files
WHERE source_file = 'leave_requests.json'
LIMIT 1
""", conn)

leave_data = json.loads(raw.loc[0, "raw_content"])
leaves = pd.json_normalize(leave_data)

print("Total leave requests:", len(leaves))

print("\n1. Leave types")
display(leaves["leave.type"].value_counts(dropna=False).rename_axis("leave_type").reset_index(name="count"))

print("\n2. Missing data")
display(
    leaves[[
        "employee.emp_id", "leave.from", "leave.to",
        "leave.days_requested", "cancelled_at", "approvals"
    ]].isna().sum().rename("missing_count").reset_index(names="column")
)

print("\n3. Approval examples")
for i, approval in enumerate(leaves["approvals"].head(3), start=1):
    print(f"\nRequest {i}:")
    print(json.dumps(approval, indent=2)[:1500])

print("\n4. Approval status counts")
approval_rows = []

for approval in leaves["approvals"]:
    if isinstance(approval, list):
        for step in approval:
            if isinstance(step, dict):
                approval_rows.append(step)

if approval_rows:
    approval_df = pd.json_normalize(approval_rows)
    print("Approval columns:", approval_df.columns.tolist())

    status_cols = [
        col for col in approval_df.columns
        if any(word in col.lower() for word in
               ["status", "decision", "approved", "state"])
    ]

    for col in status_cols:
        print(f"\n{col}:")
        display(
            approval_df[col].value_counts(dropna=False)
            .rename_axis(col).reset_index(name="count")
        )

conn.close()

Total leave requests: 5372

1. Leave types


,leave_type,count
0,Annual,2703
1,Casual,1611
2,Sick,1058



2. Missing data


TypeError: Series.reset_index() got an unexpected keyword argument 'names'. Did you mean 'name'?

In [94]:

missing = leaves[[
    "employee.emp_id", "leave.from", "leave.to",
    "leave.days_requested", "cancelled_at", "approvals"
]].isna().sum().reset_index()

missing.columns = ["column", "missing_count"]
display(missing)

,column,missing_count
0,employee.emp_id,0
1,leave.from,0
2,leave.to,0
3,leave.days_requested,0
4,cancelled_at,5118
5,approvals,0


In [96]:
import sqlite3, json
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

raw = pd.read_sql_query("""
SELECT raw_content
FROM raw_json_files
WHERE source_file = 'leave_requests.json'
LIMIT 1
""", conn)

leaves = pd.json_normalize(json.loads(raw.loc[0, "raw_content"]))

approval_rows = []
for _, row in leaves.iterrows():
    approvals = row["approvals"]
    if isinstance(approvals, str):
        try:
            approvals = json.loads(approvals)
        except Exception:
            continue

    if isinstance(approvals, list):
        for step in approvals:
            if isinstance(step, dict):
                approval_rows.append(step)

approval_df = pd.json_normalize(approval_rows)

print("Approval columns:")
print(approval_df.columns.tolist())

for col in approval_df.columns:
    if any(word in col.lower() for word in
           ["status", "decision", "approved", "state"]):
        print(f"\nValues in {col}:")
        display(
            approval_df[col].value_counts(dropna=False)
            .rename_axis("status").reset_index(name="count")
        )

print("\nFirst 3 approval records:")
display(approval_df.head(3))

conn.close()

Approval columns:
['step', 'approver_role', 'approver_id', 'action', 'acted_at']

First 3 approval records:


,step,approver_role,approver_id,action,acted_at
0,1,Manager,E10827,Approved,2026-04-05T02:10:00
1,2,HR,HR-DESK,Approved,2026-04-05T07:10:00
2,1,Manager,E10827,Approved,2026-04-15T20:18:00


In [97]:
import sqlite3
import json
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

# Load raw leave JSON
raw = pd.read_sql_query("""
    SELECT raw_content
    FROM raw_json_files
    WHERE source_file = 'leave_requests.json'
    LIMIT 1
""", conn)

leave_data = json.loads(raw.loc[0, "raw_content"])
leaves = pd.json_normalize(leave_data)

def classify_leave(row):
    if pd.notna(row.get("cancelled_at")) and str(row["cancelled_at"]).strip():
        return "Cancelled"

    approvals = row.get("approvals", [])
    if isinstance(approvals, str):
        try:
            approvals = json.loads(approvals)
        except Exception:
            approvals = []

    if not isinstance(approvals, list) or len(approvals) == 0:
        return "Pending/Review"

    actions = [
        str(step.get("action", "")).strip().lower()
        for step in approvals if isinstance(step, dict)
    ]

    if not actions:
        return "Pending/Review"
    if any(action == "rejected" for action in actions):
        return "Rejected"
    if all(action == "approved" for action in actions):
        return "Approved"

    return "Pending/Review"

# Count recorded approval steps
def approval_step_count(value):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except Exception:
            return 0
    return len(value) if isinstance(value, list) else 0

stg = pd.DataFrame({
    "request_id": leaves["request_id"],
    "submitted_at": leaves["submitted_at"],
    "emp_id": leaves["employee.emp_id"],
    "plant": leaves["employee.plant"],
    "leave_type": leaves["leave.type"],
    "leave_from": leaves["leave.from"],
    "leave_to": leaves["leave.to"],
    "days_requested": pd.to_numeric(
        leaves["leave.days_requested"], errors="coerce"
    ),
    "cancelled_at": leaves["cancelled_at"],
    "approval_steps": leaves["approvals"].apply(approval_step_count),
    "leave_status": leaves.apply(classify_leave, axis=1)
})

# Keep date fields consistent
for col in ["leave_from", "leave_to"]:
    stg[col] = pd.to_datetime(stg[col], errors="coerce").dt.strftime("%Y-%m-%d")

stg.to_sql("stg_leave_requests", conn, if_exists="replace", index=False)

print("Staging table created: stg_leave_requests")
print("Total requests:", len(stg))
print("\nLeave status summary:")
display(
    stg["leave_status"].value_counts(dropna=False)
    .rename_axis("leave_status")
    .reset_index(name="request_count")
)

print("\nData quality checks:")
print("Duplicate request IDs:", stg["request_id"].duplicated().sum())
print("Missing employee IDs:", stg["emp_id"].isna().sum())
print("Missing leave days:", stg["days_requested"].isna().sum())
print("Invalid date ranges:", (stg["leave_from"] > stg["leave_to"]).sum())

conn.close()

Staging table created: stg_leave_requests
Total requests: 5372

Leave status summary:


,leave_status,request_count
0,Approved,4434
1,Rejected,684
2,Cancelled,254



Data quality checks:
Duplicate request IDs: 0
Missing employee IDs: 0
Missing leave days: 0
Invalid date ranges: 0


In [98]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

for table in ["raw_overtime_requests", "raw_payroll_lines"]:
    print(f"\n{'='*20} {table} {'='*20}")

    print("Columns:")
    display(pd.read_sql_query(f"PRAGMA table_info({table})", conn)[
        ["name", "type"]
    ])

    print("Sample records:")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn))

    print("Row count:")
    print(pd.read_sql_query(
        f"SELECT COUNT(*) AS total_rows FROM {table}", conn
    ).iloc[0, 0])

conn.close()


==================== raw_overtime_requests ====================
Columns:


,name,type
0,request_id,TEXT
1,emp_id,TEXT
2,ot_date,TEXT
3,requested_hours,TEXT
4,approved_hours,TEXT
5,status,TEXT
6,approver_id,TEXT
7,requested_at,TEXT
8,_source_file,TEXT
9,_loaded_at,TEXT


Sample records:


,request_id,emp_id,ot_date,requested_hours,approved_hours,status,approver_id,requested_at,_source_file,_loaded_at,_row_hash
0,OT000001,E10001,2026-04-03,2.0,2.0,Approved,E11046,2026-04-04T14:07:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,7474608183123249103
1,OT000002,E10001,2026-04-14,3.0,3.0,Approved,E10827,2026-04-14T11:02:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,13681562898068836583
2,OT000003,E10001,2026-04-25,1.5,1.5,Approved,E10827,2026-04-26T13:04:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,11419779759408746226
3,OT000004,E10001,2026-04-27,1.5,,Rejected,E11046,2026-04-29T13:56:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,17314775170090654209
4,OT000005,E10001,2026-04-30,1.5,1.5,Approved,E10827,2026-04-30T09:41:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,15946472562581971434


Row count:
22098

==================== raw_payroll_lines ====================
Columns:


,name,type
0,line_id,TEXT
1,payroll_month,TEXT
2,emp_id,TEXT
3,component,TEXT
4,component_type,TEXT
5,units,TEXT
6,amount,TEXT
7,_source_file,TEXT
8,_loaded_at,TEXT
9,_row_hash,TEXT


Sample records:


,line_id,payroll_month,emp_id,component,component_type,units,amount,_source_file,_loaded_at,_row_hash
0,PL0000001,2026-04,E10001,BASIC,EARNING,,39000,payroll_lines.csv,2026-10-24T20:30:33.476774+00:00,2368814511148583083
1,PL0000002,2026-04,E10001,HOUSE_RENT,EARNING,,17550,payroll_lines.csv,2026-10-24T20:30:33.476774+00:00,4998532546379896454
2,PL0000003,2026-04,E10001,MEDICAL,EARNING,,3900,payroll_lines.csv,2026-10-24T20:30:33.476774+00:00,547623220086626455
3,PL0000004,2026-04,E10001,CONVEYANCE,EARNING,,3000,payroll_lines.csv,2026-10-24T20:30:33.476774+00:00,10515238154798638775
4,PL0000005,2026-04,E10001,OVERTIME,EARNING,8.0,3000,payroll_lines.csv,2026-10-24T20:30:33.476774+00:00,3823903304637912126


Row count:
53617


In [99]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("Overtime columns:")
display(pd.read_sql_query(
    "PRAGMA table_info(raw_overtime_requests)", conn
)[["name", "type"]])

print("Sample overtime requests:")
display(pd.read_sql_query(
    "SELECT * FROM raw_overtime_requests LIMIT 8", conn
))

conn.close()

Overtime columns:


,name,type
0,request_id,TEXT
1,emp_id,TEXT
2,ot_date,TEXT
3,requested_hours,TEXT
4,approved_hours,TEXT
5,status,TEXT
6,approver_id,TEXT
7,requested_at,TEXT
8,_source_file,TEXT
9,_loaded_at,TEXT


Sample overtime requests:


,request_id,emp_id,ot_date,requested_hours,approved_hours,status,approver_id,requested_at,_source_file,_loaded_at,_row_hash
0,OT000001,E10001,2026-04-03,2.0,2.0,Approved,E11046,2026-04-04T14:07:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,7474608183123249103
1,OT000002,E10001,2026-04-14,3.0,3.0,Approved,E10827,2026-04-14T11:02:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,13681562898068836583
2,OT000003,E10001,2026-04-25,1.5,1.5,Approved,E10827,2026-04-26T13:04:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,11419779759408746226
3,OT000004,E10001,2026-04-27,1.5,,Rejected,E11046,2026-04-29T13:56:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,17314775170090654209
4,OT000005,E10001,2026-04-30,1.5,1.5,Approved,E10827,2026-04-30T09:41:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,15946472562581971434
5,OT000006,E10001,2026-05-12,2.0,2.0,Approved,E11046,2026-05-12T10:40:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,3189274404886651406
6,OT000007,E10001,2026-05-15,1.0,1.0,Approved,E10827,2026-05-15T10:00:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,15292117958846469743
7,OT000008,E10001,2026-06-01,3.0,3.0,Approved,E10827,2026-06-03T14:27:00,overtime_requests.csv,2026-10-24T20:30:32.901776+00:00,16343988537721432039


In [100]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("peopleops_hackathon.db")

# 1. Load overtime requests
ot = pd.read_sql_query("SELECT * FROM raw_overtime_requests", conn)

ot["ot_date"] = pd.to_datetime(ot["ot_date"], errors="coerce")
ot["requested_hours"] = pd.to_numeric(ot["requested_hours"], errors="coerce")
ot["approved_hours"] = pd.to_numeric(ot["approved_hours"], errors="coerce")
ot["status_clean"] = ot["status"].astype(str).str.strip().str.title()
ot["payroll_month"] = ot["ot_date"].dt.strftime("%Y-%m")

# Create cleaned overtime staging table
ot.to_sql("stg_overtime_requests", conn, if_exists="replace", index=False)

# 2. Summarize approved overtime by employee and month
approved = ot[
    ot["status_clean"].eq("Approved")
].groupby(["emp_id", "payroll_month"], as_index=False).agg(
    approved_requests=("request_id", "nunique"),
    approved_ot_hours=("approved_hours", "sum")
)

# 3. Load and clean payroll
pay = pd.read_sql_query("SELECT * FROM raw_payroll_lines", conn)
pay["amount_num"] = pd.to_numeric(pay["amount"], errors="coerce")
pay["units_num"] = pd.to_numeric(pay["units"], errors="coerce")
pay["component_clean"] = pay["component"].astype(str).str.strip().str.upper()
pay["component_type_clean"] = pay["component_type"].astype(str).str.strip().str.upper()

# Create cleaned payroll staging table
pay.to_sql("stg_payroll_lines", conn, if_exists="replace", index=False)

# 4. Aggregate overtime payroll by employee and month
paid_ot = pay[
    pay["component_clean"].eq("OVERTIME")
].groupby(["emp_id", "payroll_month"], as_index=False).agg(
    overtime_pay_lines=("line_id", "nunique"),
    paid_ot_hours=("units_num", "sum"),
    overtime_amount=("amount_num", "sum")
)

# 5. Reconcile approved OT against paid OT
recon = approved.merge(
    paid_ot,
    on=["emp_id", "payroll_month"],
    how="outer"
)

recon["approved_requests"] = recon["approved_requests"].fillna(0).astype(int)
recon["approved_ot_hours"] = recon["approved_ot_hours"].fillna(0)
recon["overtime_pay_lines"] = recon["overtime_pay_lines"].fillna(0).astype(int)
recon["paid_ot_hours"] = recon["paid_ot_hours"].fillna(0)
recon["overtime_amount"] = recon["overtime_amount"].fillna(0)

recon["hours_difference"] = (
    recon["paid_ot_hours"] - recon["approved_ot_hours"]
).round(2)

recon["reconciliation_status"] = np.select(
    [
        (recon["approved_ot_hours"] > 0) & (recon["paid_ot_hours"] == 0),
        (recon["approved_ot_hours"] == 0) & (recon["paid_ot_hours"] > 0),
        recon["hours_difference"].abs() > 0.01
    ],
    [
        "Approved OT not found in payroll",
        "Payroll OT without approved request",
        "Hours mismatch"
    ],
    default="Hours matched"
)

recon.to_sql("rpt_overtime_payroll_reconciliation",
             conn, if_exists="replace", index=False)

# 6. Print results
print("Overtime staging rows:", len(ot))
print("Payroll staging rows:", len(pay))
print("Payroll overtime lines:", int(pay["component_clean"].eq("OVERTIME").sum()))

print("\nOvertime request statuses:")
display(
    ot["status_clean"].value_counts(dropna=False)
    .rename_axis("status").reset_index(name="count")
)

print("\nReconciliation status:")
display(
    recon["reconciliation_status"].value_counts()
    .rename_axis("status").reset_index(name="employee_months")
)

print("\nOverall totals:")
display(pd.DataFrame([{
    "approved_OT_hours": round(recon["approved_ot_hours"].sum(), 2),
    "paid_OT_hours": round(recon["paid_ot_hours"].sum(), 2),
    "paid_OT_amount": round(recon["overtime_amount"].sum(), 2),
    "hours_difference_paid_minus_approved": round(recon["hours_difference"].sum(), 2),
    "employee_month_combinations": len(recon)
}]))

print("\nLargest mismatches:")
display(
    recon.loc[recon["hours_difference"].abs().sort_values(ascending=False).index]
    .head(10)[[
        "emp_id", "payroll_month", "approved_ot_hours",
        "paid_ot_hours", "hours_difference",
        "overtime_amount", "reconciliation_status"
    ]]
)

conn.close()

Overtime staging rows: 22098
Payroll staging rows: 53617
Payroll overtime lines: 6572

Overtime request statuses:


,status,count
0,Approved,19666
1,Rejected,1346
2,Pending,1086



Reconciliation status:


,status,employee_months
0,Hours matched,6572



Overall totals:


,approved_OT_hours,paid_OT_hours,paid_OT_amount,hours_difference_paid_minus_approved,employee_month_combinations
0,36401.5,36401.5,19003890,0.0,6572



Largest mismatches:


,emp_id,payroll_month,approved_ot_hours,paid_ot_hours,hours_difference,overtime_amount,reconciliation_status
0,E10001,2026-04,8.0,8.0,0.0,3000,Hours matched
1,E10001,2026-05,3.0,3.0,0.0,1125,Hours matched
2,E10001,2026-06,3.0,3.0,0.0,1125,Hours matched
3,E10002,2026-04,9.5,9.5,0.0,3471,Hours matched
4,E10002,2026-05,7.0,7.0,0.0,2558,Hours matched
5,E10002,2026-07,1.0,1.0,0.0,404,Hours matched
6,E10002,2026-08,4.0,4.0,0.0,1615,Hours matched
7,E10002,2026-09,16.5,16.5,0.0,6663,Hours matched
8,E10003,2026-04,2.5,2.5,0.0,1190,Hours matched
9,E10003,2026-05,2.0,2.0,0.0,952,Hours matched


In [101]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports", exist_ok=True)

def save_report(name, query):
    df = pd.read_sql_query(query, conn)
    df.to_csv(f"final_reports/{name}.csv", index=False)
    print(f"{name}.csv — {len(df):,} rows")
    display(df.head(10))
    return df

print("REPORT 1: Monthly payroll by component")
monthly_payroll = save_report(
    "01_monthly_payroll_by_component",
    """
    SELECT payroll_month, component, component_type,
           COUNT(*) AS line_count,
           SUM(CAST(amount AS REAL)) AS total_amount,
           SUM(CAST(units AS REAL)) AS total_units
    FROM raw_payroll_lines
    GROUP BY payroll_month, component, component_type
    ORDER BY payroll_month, component
    """
)

print("\nREPORT 2: Monthly overtime reconciliation")
ot_recon = save_report(
    "02_overtime_reconciliation",
    """
    SELECT payroll_month, reconciliation_status,
           COUNT(*) AS employee_months,
           SUM(approved_ot_hours) AS approved_hours,
           SUM(paid_ot_hours) AS paid_hours,
           SUM(overtime_amount) AS overtime_amount,
           SUM(hours_difference) AS net_hours_difference
    FROM rpt_overtime_payroll_reconciliation
    GROUP BY payroll_month, reconciliation_status
    ORDER BY payroll_month, reconciliation_status
    """
)

print("\nREPORT 3: Leave by type and status")
leave_summary = save_report(
    "03_leave_summary",
    """
    SELECT leave_type, leave_status,
           COUNT(*) AS request_count,
           SUM(days_requested) AS requested_days
    FROM stg_leave_requests
    GROUP BY leave_type, leave_status
    ORDER BY leave_type, leave_status
    """
)

print("\nREPORT 4: Payroll totals by department")
dept_columns = pd.read_sql_query(
    "PRAGMA table_info(stg_employees_departments)", conn
)["name"].tolist()

if "department_clean" in dept_columns:
    dept_payroll = save_report(
        "04_department_payroll",
        """
        SELECT e.department_clean,
               p.payroll_month,
               COUNT(DISTINCT p.emp_id) AS employees,
               SUM(CASE WHEN UPPER(p.component_type) = 'EARNING'
                        THEN CAST(p.amount AS REAL) ELSE 0 END) AS earnings,
               SUM(CASE WHEN UPPER(p.component_type) = 'DEDUCTION'
                        THEN CAST(p.amount AS REAL) ELSE 0 END) AS deductions,
               SUM(CAST(p.amount AS REAL)) AS net_component_sum
        FROM raw_payroll_lines p
        LEFT JOIN stg_employees_departments e
          ON p.emp_id = e.emp_id
        GROUP BY e.department_clean, p.payroll_month
        ORDER BY p.payroll_month, earnings DESC
        """
    )
else:
    print("Skipped: department_clean column not found.")

print("\nREPORT 5: Attendance staging quality")
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)["name"].tolist()

if "stg_biometric_punches" in tables:
    attendance_quality = save_report(
        "05_attendance_data_quality",
        """
        SELECT DATE(punch_time) AS punch_date,
               COUNT(*) AS mapped_punches,
               COUNT(DISTINCT emp_id) AS employees
        FROM stg_biometric_punches
        GROUP BY DATE(punch_time)
        ORDER BY punch_date
        """
    )

if "dq_unmatched_punches" in tables:
    unmatched = save_report(
        "06_unmatched_punches",
        "SELECT * FROM dq_unmatched_punches"
    )

print("\nExport folder:", os.path.abspath("final_reports"))
print("Reports created successfully.")
conn.close()

REPORT 1: Monthly payroll by component
01_monthly_payroll_by_component.csv — 54 rows


,payroll_month,component,component_type,line_count,total_amount,total_units
0,2026-04,ABSENCE_DEDUCTION,DEDUCTION,553,1682495.0,839.0
1,2026-04,BASIC,EARNING,1213,66840600.0,0.0
2,2026-04,CONVEYANCE,EARNING,1213,4527866.0,0.0
3,2026-04,EOBI,DEDUCTION,1213,485200.0,0.0
4,2026-04,HOUSE_RENT,EARNING,1213,30078272.0,0.0
5,2026-04,INCOME_TAX,DEDUCTION,1201,2714218.0,0.0
6,2026-04,LOAN_INSTALLMENT,DEDUCTION,84,804000.0,0.0
7,2026-04,MEDICAL,EARNING,1213,6684060.0,0.0
8,2026-04,OVERTIME,EARNING,1119,2814309.0,5525.5
9,2026-05,ABSENCE_DEDUCTION,DEDUCTION,520,1470823.0,708.0



REPORT 2: Monthly overtime reconciliation
02_overtime_reconciliation.csv — 6 rows


,payroll_month,reconciliation_status,employee_months,approved_hours,paid_hours,overtime_amount,net_hours_difference
0,2026-04,Hours matched,1119,5525.5,5525.5,2814309,0.0
1,2026-05,Hours matched,1081,5806.5,5806.5,2855346,0.0
2,2026-06,Hours matched,1101,5889.0,5889.0,2925658,0.0
3,2026-07,Hours matched,1095,6249.0,6249.0,3366750,0.0
4,2026-08,Hours matched,1086,6513.0,6513.0,3566669,0.0
5,2026-09,Hours matched,1090,6418.5,6418.5,3475158,0.0



REPORT 3: Leave by type and status
03_leave_summary.csv — 9 rows


...


REPORT 5: Attendance staging quality
05_attendance_data_quality.csv — 184 rows


,punch_date,mapped_punches,employees
0,2026-04-01,2034,1134
1,2026-04-02,2331,1137
2,2026-04-03,2312,1123
3,2026-04-04,2277,1106
4,2026-04-05,364,324
5,2026-04-06,1964,1119
6,2026-04-07,2313,1134
7,2026-04-08,2271,1123
8,2026-04-09,2275,1117
9,2026-04-10,2329,1130


06_unmatched_punches.csv — 6,962 rows


,punch_id,device_id,device_user_id,punch_time,emp_id,full_name,department_clean
0,P00000342,DEV-FSD-01,90001,2026-04-01 07:32:45,None,None,None
1,P00000343,DEV-FSD-01,90012,2026-04-01 07:37:01,None,None,None
2,P00000344,DEV-FSD-01,90003,2026-04-01 07:41:09,None,None,None
3,P00000345,DEV-KOR-01,90027,2026-04-01 07:47:12,None,None,None
4,P00000346,DEV-FSD-01,90007,2026-04-01 07:47:13,None,None,None
5,P00000347,DEV-SITE-01,90009,2026-04-01 07:52:39,None,None,None
6,P00000348,DEV-KOR-01,90005,2026-04-01 07:52:47,None,None,None
7,P00000349,DEV-SITE-01,90034,2026-04-01 07:55:31,None,None,None
8,P00000350,DEV-KOR-01,90014,2026-04-01 07:57:39,None,None,None
9,P00000351,DEV-KOR-01,90024,2026-04-01 08:00:52,None,None,None



Export folder: C:\Users\dell\Desktop\data-hackathon-main\final_reports
Reports created successfully.


In [102]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports", exist_ok=True)

query = """
SELECT
    e.department_clean,
    p.payroll_month,
    COUNT(DISTINCT p.emp_id) AS employees,
    SUM(CASE
        WHEN UPPER(p.component_type) = 'EARNING'
        THEN CAST(p.amount AS REAL) ELSE 0
    END) AS total_earnings,
    SUM(CASE
        WHEN UPPER(p.component_type) = 'DEDUCTION'
        THEN CAST(p.amount AS REAL) ELSE 0
    END) AS total_deductions,
    SUM(CASE
        WHEN UPPER(p.component_type) = 'EARNING'
        THEN CAST(p.amount AS REAL)
        WHEN UPPER(p.component_type) = 'DEDUCTION'
        THEN -CAST(p.amount AS REAL)
        ELSE 0
    END) AS net_payroll
FROM raw_payroll_lines p
LEFT JOIN stg_employees_departments e
    ON p.emp_id = e.emp_id
GROUP BY e.department_clean, p.payroll_month
ORDER BY p.payroll_month, net_payroll DESC
"""

df = pd.read_sql_query(query, conn)
df.to_csv("final_reports/04_department_payroll.csv", index=False)

print("Corrected department payroll report saved.")
display(df.head(10))

print("\nUnmatched punch counts by device:")
unmatched = pd.read_sql_query("""
    SELECT device_id,
           COUNT(*) AS unmatched_punches,
           COUNT(DISTINCT device_user_id) AS device_users
    FROM dq_unmatched_punches
    GROUP BY device_id
    ORDER BY unmatched_punches DESC
""", conn)
display(unmatched)

conn.close()

Corrected department payroll report saved.


,department_clean,payroll_month,employees,total_earnings,total_deductions,net_payroll
0,Stitching,2026-04,151,12233962.0,507982.0,11725980.0
1,Weaving,2026-04,132,10586170.0,393454.0,10192716.0
2,Spinning,2026-04,119,9219461.0,305688.0,8913773.0
3,Dyeing,2026-04,107,8546045.0,337177.0,8208868.0
4,Maintenance,2026-04,102,8332445.0,305952.0,8026493.0
5,Quality Control,2026-04,101,8295878.0,353674.0,7942204.0
6,Finishing,2026-04,100,7905140.0,293206.0,7611934.0
7,Cutting,2026-04,80,6346666.0,236073.0,6110593.0
8,Warehouse,2026-04,79,6212395.0,289410.0,5922985.0
9,Packing,2026-04,62,4911782.0,182290.0,4729492.0



Unmatched punch counts by device:


,device_id,unmatched_punches,device_users
0,DEV-KOR-01,2876,16
1,DEV-FSD-01,2050,12
2,DEV-SITE-01,2036,12


In [103]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

query = """
SELECT
    p.device_id,
    p.device_user_id,
    COUNT(*) AS unmatched_punches,
    MIN(p.punch_time) AS first_punch,
    MAX(p.punch_time) AS last_punch,
    COUNT(m.emp_id) AS exact_mapping_rows,
    COUNT(m2.emp_id) AS mapped_on_another_device,
    MIN(m.enrolled_on) AS enrolled_on
FROM dq_unmatched_punches p
LEFT JOIN raw_device_mapping m
    ON p.device_id = m.device_id
   AND CAST(p.device_user_id AS TEXT) = CAST(m.device_user_id AS TEXT)
LEFT JOIN (
    SELECT DISTINCT device_user_id, emp_id
    FROM raw_device_mapping
) m2
    ON CAST(p.device_user_id AS TEXT) = CAST(m2.device_user_id AS TEXT)
GROUP BY p.device_id, p.device_user_id
ORDER BY unmatched_punches DESC
"""

diagnostic = pd.read_sql_query(query, conn)

print("Unmatched device-user combinations:", len(diagnostic))
display(diagnostic.head(30))

print("\nSummary by device:")
display(
    diagnostic.groupby("device_id", as_index=False).agg(
        device_users=("device_user_id", "nunique"),
        unmatched_punches=("unmatched_punches", "sum"),
        users_with_exact_mapping=(
            "exact_mapping_rows",
            lambda s: (s > 0).sum()
        ),
        users_mapped_elsewhere=(
            "mapped_on_another_device",
            lambda s: (s > 0).sum()
        )
    )
)

conn.close()

Unmatched device-user combinations: 40


,device_id,device_user_id,unmatched_punches,first_punch,last_punch,exact_mapping_rows,mapped_on_another_device,enrolled_on
0,DEV-KOR-01,90002,194,2026-04-01 08:29:16,2026-09-28 12:39:09,0,0,None
1,DEV-KOR-01,90023,194,2026-04-02 08:19:14,2026-09-30 15:46:57,0,0,None
2,DEV-KOR-01,90037,192,2026-04-01 08:44:54,2026-09-26 14:06:21,0,0,None
3,DEV-KOR-01,90040,192,2026-04-01 08:15:25,2026-09-29 15:54:05,0,0,None
4,DEV-KOR-01,90038,190,2026-04-01 08:34:53,2026-09-28 14:47:21,0,0,None
5,DEV-FSD-01,90018,186,2026-04-03 08:54:16,2026-09-30 16:32:52,0,0,None
6,DEV-SITE-01,90008,186,2026-04-01 08:39:10,2026-09-30 18:33:17,0,0,None
7,DEV-KOR-01,90024,184,2026-04-01 08:00:52,2026-09-30 15:42:00,0,0,None
8,DEV-KOR-01,90005,182,2026-04-01 07:52:47,2026-09-29 15:58:14,0,0,None
9,DEV-KOR-01,90032,182,2026-04-01 08:12:32,2026-09-28 17:06:49,0,0,None



Summary by device:


,device_id,device_users,unmatched_punches,users_with_exact_mapping,users_mapped_elsewhere
0,DEV-FSD-01,12,2050,0,0
1,DEV-KOR-01,16,2876,0,1
2,DEV-SITE-01,12,2036,0,0


In [104]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports", exist_ok=True)

# Detailed unresolved punch exceptions
exceptions = pd.read_sql_query("""
SELECT
    p.device_id,
    p.device_user_id,
    COUNT(*) AS unmatched_punches,
    MIN(p.punch_time) AS first_punch,
    MAX(p.punch_time) AS last_punch,
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM raw_device_mapping m
            WHERE m.device_id = p.device_id
              AND CAST(m.device_user_id AS TEXT)
                  = CAST(p.device_user_id AS TEXT)
        ) THEN 'Exact mapping exists - investigate join'
        WHEN EXISTS (
            SELECT 1
            FROM raw_device_mapping m
            WHERE CAST(m.device_user_id AS TEXT)
                  = CAST(p.device_user_id AS TEXT)
        ) THEN 'Mapped on another device - verify'
        ELSE 'No mapping record found'
    END AS exception_reason
FROM dq_unmatched_punches p
GROUP BY p.device_id, p.device_user_id
ORDER BY unmatched_punches DESC
""", conn)

exceptions.to_csv(
    "final_reports/07_unmatched_device_user_exceptions.csv",
    index=False
)

print("Exception combinations:", len(exceptions))
print("\nException reasons:")
display(
    exceptions.groupby("exception_reason", as_index=False).agg(
        device_user_combinations=("device_user_id", "count"),
        unmatched_punches=("unmatched_punches", "sum")
    )
)

print("\nCross-device mapping details:")
display(pd.read_sql_query("""
SELECT DISTINCT
    p.device_id AS punch_device,
    p.device_user_id,
    m.device_id AS mapped_device,
    m.emp_id,
    m.enrolled_on
FROM dq_unmatched_punches p
JOIN raw_device_mapping m
  ON CAST(p.device_user_id AS TEXT)
     = CAST(m.device_user_id AS TEXT)
WHERE p.device_id <> m.device_id
ORDER BY p.device_user_id
""", conn))

print("\nSaved: final_reports/07_unmatched_device_user_exceptions.csv")
conn.close()

Exception combinations: 40

Exception reasons:


,exception_reason,device_user_combinations,unmatched_punches
0,Mapped on another device - verify,1,178
1,No mapping record found,39,6784



Cross-device mapping details:


,punch_device,device_user_id,mapped_device,emp_id,enrolled_on
0,DEV-KOR-01,90020,DEV-HO-01,E11122,2024-01-01



Saved: final_reports/07_unmatched_device_user_exceptions.csv


In [105]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports/business_questions", exist_ok=True)

questions = {
    "Q01_monthly_payroll_trend": """
        SELECT payroll_month,
               SUM(CASE WHEN UPPER(component_type)='EARNING'
                        THEN CAST(amount AS REAL) ELSE 0 END) AS earnings,
               SUM(CASE WHEN UPPER(component_type)='DEDUCTION'
                        THEN CAST(amount AS REAL) ELSE 0 END) AS deductions
        FROM raw_payroll_lines
        GROUP BY payroll_month ORDER BY payroll_month
    """,

    "Q02_payroll_by_component": """
        SELECT payroll_month, component,
               SUM(CAST(amount AS REAL)) AS total_amount,
               COUNT(*) AS line_count
        FROM raw_payroll_lines
        GROUP BY payroll_month, component
        ORDER BY payroll_month, total_amount DESC
    """,

    "Q03_department_payroll": """
        SELECT e.department_clean, p.payroll_month,
               SUM(CASE WHEN UPPER(p.component_type)='EARNING'
                        THEN CAST(p.amount AS REAL) ELSE 0 END) AS earnings,
               SUM(CASE WHEN UPPER(p.component_type)='DEDUCTION'
                        THEN CAST(p.amount AS REAL) ELSE 0 END) AS deductions
        FROM raw_payroll_lines p
        LEFT JOIN stg_employees_departments e ON p.emp_id=e.emp_id
        GROUP BY e.department_clean, p.payroll_month
        ORDER BY p.payroll_month, earnings DESC
    """,

    "Q04_overtime_request_status": """
        SELECT status, COUNT(*) AS requests,
               SUM(CAST(requested_hours AS REAL)) AS requested_hours,
               SUM(CAST(approved_hours AS REAL)) AS approved_hours
        FROM raw_overtime_requests
        GROUP BY status ORDER BY requests DESC
    """,

    "Q05_monthly_paid_overtime": """
        SELECT payroll_month,
               COUNT(*) AS overtime_lines,
               SUM(CAST(units AS REAL)) AS paid_hours,
               SUM(CAST(amount AS REAL)) AS overtime_amount
        FROM raw_payroll_lines
        WHERE UPPER(component)='OVERTIME'
        GROUP BY payroll_month ORDER BY payroll_month
    """,

    "Q06_top_employees_overtime_cost": """
        SELECT emp_id,
               SUM(CAST(units AS REAL)) AS paid_overtime_hours,
               SUM(CAST(amount AS REAL)) AS overtime_amount
        FROM raw_payroll_lines
        WHERE UPPER(component)='OVERTIME'
        GROUP BY emp_id
        ORDER BY overtime_amount DESC LIMIT 20
    """,

    "Q07_leave_by_type_and_status": """
        SELECT leave_type, leave_status,
               COUNT(*) AS requests,
               SUM(days_requested) AS requested_days
        FROM stg_leave_requests
        GROUP BY leave_type, leave_status
        ORDER BY leave_type, leave_status
    """,

    "Q08_approved_leave_days_by_department": """
        SELECT e.department_clean,
               COUNT(*) AS approved_leave_requests,
               SUM(l.days_requested) AS approved_leave_days
        FROM stg_leave_requests l
        LEFT JOIN stg_employees_departments e ON l.emp_id=e.emp_id
        WHERE l.leave_status='Approved'
        GROUP BY e.department_clean
        ORDER BY approved_leave_days DESC
    """,

    "Q09_deductions_by_component": """
        SELECT payroll_month, component,
               COUNT(*) AS deduction_lines,
               SUM(CAST(amount AS REAL)) AS total_deductions
        FROM raw_payroll_lines
        WHERE UPPER(component_type)='DEDUCTION'
        GROUP BY payroll_month, component
        ORDER BY payroll_month, total_deductions DESC
    """,

    "Q10_unmatched_punches_by_device": """
        SELECT device_id,
               COUNT(*) AS unmatched_punches,
               COUNT(DISTINCT device_user_id) AS affected_device_users
        FROM dq_unmatched_punches
        GROUP BY device_id
        ORDER BY unmatched_punches DESC
    """
}

for question, sql in questions.items():
    try:
        result = pd.read_sql_query(sql, conn)
        filepath = f"final_reports/business_questions/{question}.csv"
        result.to_csv(filepath, index=False)
        print(f"{question}: {len(result)} rows saved")
        display(result.head(5))
    except Exception as error:
        print(f"{question} ERROR: {error}")

conn.close()
print("\nBusiness-question report folder created.")

Q01_monthly_payroll_trend: 6 rows saved


,payroll_month,earnings,deductions
0,2026-04,110945107.0,5685913.0
1,2026-05,110746861.0,5502962.0
2,2026-06,111460953.0,5740502.0
3,2026-07,121207135.0,6800324.0
4,2026-08,120444670.0,6542406.0


Q02_payroll_by_component: 54 rows saved


,payroll_month,component,total_amount,line_count
0,2026-04,BASIC,66840600.0,1213
1,2026-04,HOUSE_RENT,30078272.0,1213
2,2026-04,MEDICAL,6684060.0,1213
3,2026-04,CONVEYANCE,4527866.0,1213
4,2026-04,OVERTIME,2814309.0,1119


Q03_department_payroll: 114 rows saved


...

Q09_deductions_by_component: 24 rows saved


,payroll_month,component,deduction_lines,total_deductions
0,2026-04,INCOME_TAX,1201,2714218.0
1,2026-04,ABSENCE_DEDUCTION,553,1682495.0
2,2026-04,LOAN_INSTALLMENT,84,804000.0
3,2026-04,EOBI,1213,485200.0
4,2026-05,INCOME_TAX,1194,2731439.0


Q10_unmatched_punches_by_device: 3 rows saved


,device_id,unmatched_punches,affected_device_users
0,DEV-KOR-01,2876,16
1,DEV-FSD-01,2050,12
2,DEV-SITE-01,2036,12



Business-question report folder created.


In [106]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. Original roster structure")
display(pd.read_sql_query(
    "PRAGMA table_info(raw_shift_roster)", conn
))

print("2. Original roster shift counts")
display(pd.read_sql_query("""
    SELECT shift_code, COUNT(*) AS roster_records,
           COUNT(DISTINCT emp_id) AS employees,
           MIN(week_start) AS first_week,
           MAX(week_start) AS last_week
    FROM raw_shift_roster
    GROUP BY shift_code
    ORDER BY shift_code
""", conn))

print("3. Expanded roster by weekday")
display(pd.read_sql_query("""
    SELECT
        strftime('%w', attendance_date) AS weekday_number,
        COUNT(*) AS scheduled_records,
        COUNT(DISTINCT emp_id) AS employees
    FROM stg_daily_roster
    GROUP BY weekday_number
    ORDER BY weekday_number
""", conn))

print("4. Holiday dates and scope")
display(pd.read_sql_query("""
    SELECT holiday_date, holiday_name, scope
    FROM raw_holidays
    ORDER BY holiday_date
""", conn))

conn.close()

1. Original roster structure


,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,TEXT,0,None,0
1,1,week_start,TEXT,0,None,0
2,2,shift_code,TEXT,0,None,0
3,3,_source_file,TEXT,0,None,0
4,4,_loaded_at,TEXT,0,None,0
5,5,_row_hash,TEXT,0,None,0


2. Original roster shift counts


,shift_code,roster_records,employees,first_week,last_week
0,DAY,10596,1108,2026-03-30,2026-10-26
1,EVE,10606,1105,2026-03-30,2026-10-26
2,GEN,4608,158,2026-03-30,2026-10-26
3,NIGHT,10592,1108,2026-03-30,2026-10-26


3. Expanded roster by weekday


,weekday_number,scheduled_records,employees
0,0,36402,1266
1,1,36402,1266
2,2,36402,1266
3,3,36402,1266
4,4,36402,1266
5,5,36402,1266
6,6,36402,1266


4. Holiday dates and scope


,holiday_date,holiday_name,scope
0,2026-04-25,Local holiday,FSD
1,2026-05-01,Labour Day,ALL
2,2026-05-27,Eid ul Adha,ALL
3,2026-05-28,Eid ul Adha,ALL
4,2026-05-29,Eid ul Adha,ALL
5,2026-06-13,Boiler inspection shutdown,SITE
6,2026-06-25,Ashura (9 Muharram),ALL
7,2026-06-26,Ashura (10 Muharram),ALL
8,2026-07-18,Plant maintenance shutdown,FSD
9,2026-08-14,Independence Day,ALL


In [107]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. Week-start weekday check")
display(pd.read_sql_query("""
    SELECT strftime('%w', week_start) AS weekday_number,
           COUNT(*) AS roster_records,
           MIN(week_start) AS first_week,
           MAX(week_start) AS last_week
    FROM raw_shift_roster
    GROUP BY weekday_number
    ORDER BY weekday_number
""", conn))

print("2. Duplicate employee-week records")
display(pd.read_sql_query("""
    SELECT emp_id, week_start, COUNT(*) AS records,
           GROUP_CONCAT(shift_code, ', ') AS shift_codes
    FROM raw_shift_roster
    GROUP BY emp_id, week_start
    HAVING COUNT(*) > 1
    ORDER BY records DESC
    LIMIT 30
""", conn))

print("3. Employees with multiple shift codes across consecutive weeks")
display(pd.read_sql_query("""
    WITH weekly AS (
        SELECT emp_id, week_start, shift_code,
               LAG(shift_code) OVER (
                   PARTITION BY emp_id ORDER BY week_start
               ) AS previous_shift
        FROM raw_shift_roster
    )
    SELECT emp_id, week_start, previous_shift, shift_code
    FROM weekly
    WHERE previous_shift IS NOT NULL
      AND previous_shift <> shift_code
    ORDER BY emp_id, week_start
    LIMIT 30
""", conn))

print("4. Employee master columns (to identify plant/location)")
display(pd.read_sql_query(
    "PRAGMA table_info(stg_employees_departments)", conn
))

conn.close()

1. Week-start weekday check


,weekday_number,roster_records,first_week,last_week
0,1,36402,2026-03-30,2026-10-26


2. Duplicate employee-week records


,emp_id,week_start,records,shift_codes


3. Employees with multiple shift codes across consecutive weeks


,emp_id,week_start,previous_shift,shift_code
0,E10001,2026-04-06,DAY,EVE
1,E10001,2026-04-13,EVE,NIGHT
2,E10001,2026-04-20,NIGHT,DAY
3,E10001,2026-04-27,DAY,EVE
4,E10001,2026-05-04,EVE,NIGHT
5,E10001,2026-05-11,NIGHT,DAY
6,E10001,2026-05-18,DAY,EVE
7,E10001,2026-05-25,EVE,NIGHT
8,E10001,2026-06-01,NIGHT,DAY
9,E10002,2026-04-06,NIGHT,DAY


4. Employee master columns (to identify plant/location)


,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,TEXT,0,None,0
1,1,full_name,TEXT,0,None,0
2,2,gender,TEXT,0,None,0
3,3,date_of_birth,TEXT,0,None,0
4,4,email,TEXT,0,None,0
5,5,phone,TEXT,0,None,0
6,6,plant,TEXT,0,None,0
7,7,department,TEXT,0,None,0
8,8,designation,TEXT,0,None,0
9,9,grade,TEXT,0,None,0


In [108]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. Employee count by plant")
display(pd.read_sql_query("""
    SELECT plant, COUNT(*) AS employees
    FROM stg_employees_departments
    GROUP BY plant
    ORDER BY employees DESC
""", conn))

print("2. Punch date coverage")
display(pd.read_sql_query("""
    SELECT MIN(date(punch_time)) AS first_punch_date,
           MAX(date(punch_time)) AS last_punch_date,
           COUNT(*) AS total_mapped_punches
    FROM stg_biometric_punches
""", conn))

print("3. Daily roster coverage")
display(pd.read_sql_query("""
    SELECT MIN(attendance_date) AS first_roster_date,
           MAX(attendance_date) AS last_roster_date,
           COUNT(DISTINCT attendance_date) AS roster_days,
           COUNT(DISTINCT emp_id) AS employees
    FROM stg_daily_roster
""", conn))

print("4. Plant-specific holiday scope compatibility")
display(pd.read_sql_query("""
    SELECT e.plant, h.scope,
           COUNT(DISTINCT h.holiday_date) AS holiday_dates
    FROM stg_employees_departments e
    CROSS JOIN raw_holidays h
    GROUP BY e.plant, h.scope
    ORDER BY e.plant, h.scope
""", conn))

print("5. Employment status and separation-date quality")
display(pd.read_sql_query("""
    SELECT status,
           COUNT(*) AS employees,
           SUM(CASE WHEN separation_date IS NOT NULL
                     AND TRIM(separation_date) <> ''
                    THEN 1 ELSE 0 END) AS with_separation_date,
           SUM(CASE WHEN hire_date_clean IS NOT NULL
                     AND separation_date IS NOT NULL
                     AND date(separation_date) < date(hire_date_clean)
                    THEN 1 ELSE 0 END) AS separation_before_hire
    FROM stg_employees_departments
    GROUP BY status
""", conn))

conn.close()

1. Employee count by plant


,plant,employees
0,Korangi Plant,399
1,Faisalabad Plant,392
2,SITE Plant,347
3,Head Office,128


2. Punch date coverage


,first_punch_date,last_punch_date,total_mapped_punches
0,2026-04-01,2026-10-01,338318


3. Daily roster coverage


,first_roster_date,last_roster_date,roster_days,employees
0,2026-03-30,2026-11-01,217,1266


4. Plant-specific holiday scope compatibility


,plant,scope,holiday_dates
0,Faisalabad Plant,ALL,8
1,Faisalabad Plant,FSD,2
2,Faisalabad Plant,KOR,1
3,Faisalabad Plant,SITE,1
4,Head Office,ALL,8
5,Head Office,FSD,2
6,Head Office,KOR,1
7,Head Office,SITE,1
8,Korangi Plant,ALL,8
9,Korangi Plant,FSD,2


5. Employment status and separation-date quality


,status,employees,with_separation_date,separation_before_hire
0,Active,1176,0,0
1,Separated,90,90,0


In [109]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports", exist_ok=True)

# Confirm the actual schema before calculating attendance
for table in [
    "stg_shift_punch_summary",
    "stg_daily_roster",
    "stg_employees_departments",
    "raw_holidays"
]:
    print(f"\n--- {table} ---")
    display(pd.read_sql_query(
        f"PRAGMA table_info({table})", conn
    ))

conn.close()


--- stg_shift_punch_summary ---


,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,TEXT,0,None,0
1,1,attendance_date,TEXT,0,None,0
2,2,shift_code,TEXT,0,None,0
3,3,shift_start,TEXT,0,None,0
4,4,shift_end,TEXT,0,None,0
5,5,grace_minutes,INT,0,None,0
6,6,actual_in,,0,None,0
7,7,actual_out,,0,None,0
8,8,punch_count,,0,None,0



--- stg_daily_roster ---


,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,TEXT,0,None,0
1,1,attendance_date,TEXT,0,None,0
2,2,shift_code,TEXT,0,None,0
3,3,shift_start,TEXT,0,None,0
4,4,shift_end,TEXT,0,None,0
5,5,grace_minutes,INTEGER,0,None,0



--- stg_employees_departments ---


,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,TEXT,0,None,0
1,1,full_name,TEXT,0,None,0
2,2,gender,TEXT,0,None,0
3,3,date_of_birth,TEXT,0,None,0
4,4,email,TEXT,0,None,0
5,5,phone,TEXT,0,None,0
6,6,plant,TEXT,0,None,0
7,7,department,TEXT,0,None,0
8,8,designation,TEXT,0,None,0
9,9,grade,TEXT,0,None,0



--- raw_holidays ---


,cid,name,type,notnull,dflt_value,pk
0,0,holiday_date,TEXT,0,None,0
1,1,holiday_name,TEXT,0,None,0
2,2,scope,TEXT,0,None,0
3,3,_source_file,TEXT,0,None,0
4,4,_loaded_at,TEXT,0,None,0
5,5,_row_hash,TEXT,0,None,0


In [112]:


import sqlite3
import pandas as pd
import os

conn = sqlite3.connect("peopleops_hackathon.db")
os.makedirs("final_reports", exist_ok=True)

query = """
WITH daily AS (
    SELECT
        r.emp_id,
        r.attendance_date,
        r.shift_code,
        r.shift_start,
        r.shift_end,
        r.grace_minutes,
        COALESCE(s.punch_count, 0) AS punch_count,
        s.actual_in,
        s.actual_out,
        e.full_name,
        e.plant,
        e.department_clean,
        e.status AS employment_status,
        h.holiday_name,
        CASE WHEN h.holiday_name IS NOT NULL THEN 1 ELSE 0 END
            AS is_holiday
    FROM stg_daily_roster r
    JOIN stg_employees_departments e
      ON e.emp_id = r.emp_id
    LEFT JOIN stg_shift_punch_summary s
      ON s.emp_id = r.emp_id
     AND s.attendance_date = r.attendance_date
     AND s.shift_code = r.shift_code
    LEFT JOIN raw_holidays h
      ON h.holiday_date = r.attendance_date
     AND (
          h.scope = 'ALL'
          OR (h.scope = 'FSD' AND e.plant = 'Faisalabad Plant')
          OR (h.scope = 'KOR' AND e.plant = 'Korangi Plant')
          OR (h.scope = 'SITE' AND e.plant = 'SITE Plant')
     )
)
SELECT *,
    CASE
        WHEN punch_count >= 2 AND is_holiday = 1
            THEN 'Holiday Worked'
        WHEN punch_count >= 2
            THEN 'Present'
        WHEN punch_count = 1
            THEN 'Missing Punch'
        WHEN is_holiday = 1
            THEN 'Holiday'
        ELSE 'No Punch Recorded'
    END AS provisional_status,
    CASE
        WHEN actual_in IS NOT NULL
         AND datetime(actual_in) >
             datetime(shift_start, '+' || grace_minutes || ' minutes')
            THEN 1
        ELSE 0
    END AS late_over_grace
FROM daily
ORDER BY attendance_date, emp_id
"""

try:
    attendance = pd.read_sql_query(query, conn)

    attendance.to_csv(
        "final_reports/08_provisional_daily_attendance.csv",
        index=False
    )

    summary = (
        attendance.groupby("provisional_status")
        .size()
        .reset_index(name="employee_shift_records")
    )
    summary.to_csv(
        "final_reports/09_provisional_attendance_summary.csv",
        index=False
    )

    print("Attendance rows:", len(attendance))
    print("Saved: final_reports/08_provisional_daily_attendance.csv")
    print("Saved: final_reports/09_provisional_attendance_summary.csv")

    print("\nProvisional status counts:")
    display(summary)

    print("\nLate arrivals by plant:")
    display(
        attendance.groupby("plant")["late_over_grace"]
        .agg(["sum", "count"])
        .reset_index()
        .rename(columns={
            "sum": "late_shift_records",
            "count": "scheduled_shift_records"
        })
    )

except Exception as ex:
    print("Query error:", ex)

finally:
    conn.close()



Attendance rows: 254814
Saved: final_reports/08_provisional_daily_attendance.csv
Saved: final_reports/09_provisional_attendance_summary.csv

Provisional status counts:


,provisional_status,employee_shift_records
0,Holiday,10648
1,Holiday Worked,178
2,Missing Punch,18261
3,No Punch Recorded,79728
4,Present,145999



Late arrivals by plant:


,plant,late_shift_records,scheduled_shift_records
0,Faisalabad Plant,6906,78834
1,Head Office,1015,25914
2,Korangi Plant,6697,80178
3,SITE Plant,25717,69888


In [113]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. Lateness by plant and shift")
display(pd.read_sql_query("""
    SELECT
        plant,
        shift_code,
        COUNT(*) AS scheduled_records,
        SUM(late_over_grace) AS late_records,
        ROUND(
            100.0 * SUM(late_over_grace) / COUNT(*), 2
        ) AS late_percentage
    FROM (
        SELECT a.*, e.plant
        FROM (
            SELECT
                emp_id, attendance_date, shift_code,
                late_over_grace
            FROM (
                SELECT
                    emp_id, attendance_date, shift_code,
                    CASE
                        WHEN actual_in IS NOT NULL
                         AND datetime(actual_in) >
                             datetime(
                                 shift_start,
                                 '+' || grace_minutes || ' minutes'
                             )
                        THEN 1 ELSE 0
                    END AS late_over_grace
                FROM stg_shift_punch_summary
            )
        ) a
        JOIN stg_employees_departments e
          ON a.emp_id = e.emp_id
    )
    GROUP BY plant, shift_code
    ORDER BY late_percentage DESC
""", conn))

print("2. Lateness by month and plant")
display(pd.read_sql_query("""
    SELECT
        e.plant,
        substr(s.attendance_date, 1, 7) AS month,
        COUNT(*) AS scheduled_records,
        SUM(
            CASE
                WHEN s.actual_in IS NOT NULL
                 AND datetime(s.actual_in) >
                     datetime(
                         s.shift_start,
                         '+' || s.grace_minutes || ' minutes'
                     )
                THEN 1 ELSE 0
            END
        ) AS late_records
    FROM stg_shift_punch_summary s
    JOIN stg_employees_departments e
      ON s.emp_id = e.emp_id
    GROUP BY e.plant, month
    ORDER BY month, e.plant
""", conn))

conn.close()

1. Lateness by plant and shift


,plant,shift_code,scheduled_records,late_records,late_percentage
0,SITE Plant,DAY,23240,8834,38.01
1,SITE Plant,NIGHT,23177,8804,37.99
2,SITE Plant,GEN,217,78,35.94
3,SITE Plant,EVE,23254,8001,34.41
4,Korangi Plant,NIGHT,24808,2809,11.32
5,Faisalabad Plant,NIGHT,26159,2948,11.27
6,Faisalabad Plant,DAY,26040,2796,10.74
7,Korangi Plant,DAY,24892,2662,10.69
8,Faisalabad Plant,GEN,651,56,8.60
9,Faisalabad Plant,EVE,25984,1106,4.26


2. Lateness by month and plant


,plant,month,scheduled_records,late_records
0,Faisalabad Plant,2026-03,748,0
1,Head Office,2026-03,240,0
2,Korangi Plant,2026-03,762,0
3,SITE Plant,2026-03,662,0
4,Faisalabad Plant,2026-04,11120,1143
5,Head Office,2026-04,3571,166
6,Korangi Plant,2026-04,11311,1122
7,SITE Plant,2026-04,9798,968
8,Faisalabad Plant,2026-05,11344,1137
9,Head Office,2026-05,3672,137


In [114]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. SITE Plant lateness by month and shift")
display(pd.read_sql_query("""
    SELECT
        substr(s.attendance_date, 1, 7) AS month,
        s.shift_code,
        COUNT(*) AS shifts,
        SUM(CASE WHEN s.actual_in IS NOT NULL THEN 1 ELSE 0 END)
            AS shifts_with_in_punch,
        SUM(
            CASE WHEN s.actual_in IS NOT NULL
                  AND datetime(s.actual_in) >
                      datetime(s.shift_start, '+' ||
                               s.grace_minutes || ' minutes')
                 THEN 1 ELSE 0 END
        ) AS late_shifts
    FROM stg_shift_punch_summary s
    JOIN stg_employees_departments e ON e.emp_id = s.emp_id
    WHERE e.plant = 'SITE Plant'
      AND substr(s.attendance_date, 1, 7)
          BETWEEN '2026-04' AND '2026-09'
    GROUP BY month, s.shift_code
    ORDER BY month, s.shift_code
""", conn))

print("2. Sample records from July to September marked late")
display(pd.read_sql_query("""
    SELECT
        s.emp_id, s.attendance_date, s.shift_code,
        s.shift_start, s.actual_in, s.grace_minutes,
        datetime(s.shift_start, '+' ||
                 s.grace_minutes || ' minutes') AS grace_deadline,
        s.punch_count
    FROM stg_shift_punch_summary s
    JOIN stg_employees_departments e ON e.emp_id = s.emp_id
    WHERE e.plant = 'SITE Plant'
      AND s.attendance_date BETWEEN '2026-07-01' AND '2026-09-30'
      AND s.actual_in IS NOT NULL
      AND datetime(s.actual_in) >
          datetime(s.shift_start, '+' || s.grace_minutes || ' minutes')
    ORDER BY s.attendance_date, s.emp_id
    LIMIT 20
""", conn))

conn.close()


1. SITE Plant lateness by month and shift


,month,shift_code,shifts,shifts_with_in_punch,late_shifts
0,2026-04,DAY,3220,2578,378
1,2026-04,EVE,3273,2587,190
2,2026-04,GEN,30,26,1
3,2026-04,NIGHT,3275,2880,399
4,2026-05,DAY,3400,2237,412
5,2026-05,EVE,3277,2173,168
6,2026-05,GEN,31,22,1
7,2026-05,NIGHT,3349,2508,482
8,2026-06,DAY,3198,2249,453
9,2026-06,EVE,3279,2330,141


2. Sample records from July to September marked late


,emp_id,attendance_date,shift_code,shift_start,actual_in,grace_minutes,grace_deadline,punch_count
0,E10004,2026-07-01,EVE,2026-07-01 14:00:00,2026-07-01 15:01:30,10,2026-07-01 14:10:00,2
1,E10014,2026-07-01,EVE,2026-07-01 14:00:00,2026-07-01 14:41:19,10,2026-07-01 14:10:00,2
2,E10021,2026-07-01,EVE,2026-07-01 14:00:00,2026-07-01 15:09:14,10,2026-07-01 14:10:00,2
3,E10022,2026-07-01,NIGHT,2026-07-01 22:00:00,2026-07-01 22:40:13,10,2026-07-01 22:10:00,2
4,E10024,2026-07-01,NIGHT,2026-07-01 22:00:00,2026-07-01 23:36:01,10,2026-07-01 22:10:00,2
5,E10026,2026-07-01,DAY,2026-07-01 06:00:00,2026-07-01 07:11:00,10,2026-07-01 06:10:00,2
6,E10028,2026-07-01,NIGHT,2026-07-01 22:00:00,2026-07-01 22:56:27,10,2026-07-01 22:10:00,2
7,E10033,2026-07-01,EVE,2026-07-01 14:00:00,2026-07-01 14:50:14,10,2026-07-01 14:10:00,2
8,E10036,2026-07-01,EVE,2026-07-01 14:00:00,2026-07-01 14:41:34,10,2026-07-01 14:10:00,2
9,E10037,2026-07-01,NIGHT,2026-07-01 22:00:00,2026-07-01 22:58:09,10,2026-07-01 22:10:00,2


In [115]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. Punch summary table columns")
display(pd.read_sql_query(
    "PRAGMA table_info(stg_punch_shift_assignment)", conn
))

print("2. Actual-in time and minutes after shift start by month")
display(pd.read_sql_query("""
    SELECT
        substr(attendance_date, 1, 7) AS month,
        shift_code,
        COUNT(*) AS records_with_in,
        MIN(time(actual_in)) AS earliest_in_time,
        MAX(time(actual_in)) AS latest_in_time,
        ROUND(AVG(
            (julianday(actual_in) - julianday(shift_start)) * 1440
        ), 1) AS avg_minutes_after_shift_start,
        ROUND(MIN(
            (julianday(actual_in) - julianday(shift_start)) * 1440
        ), 1) AS min_minutes_after_start,
        ROUND(MAX(
            (julianday(actual_in) - julianday(shift_start)) * 1440
        ), 1) AS max_minutes_after_start
    FROM stg_shift_punch_summary
    WHERE attendance_date BETWEEN '2026-04-01' AND '2026-09-30'
      AND actual_in IS NOT NULL
      AND emp_id IN (
          SELECT emp_id
          FROM stg_employees_departments
          WHERE plant = 'SITE Plant'
      )
    GROUP BY month, shift_code
    ORDER BY month, shift_code
""", conn))

conn.close()


1. Punch summary table columns


,cid,name,type,notnull,dflt_value,pk
0,0,punch_id,TEXT,0,None,0
1,1,punch_time,TEXT,0,None,0
2,2,emp_id,TEXT,0,None,0
3,3,attendance_date,TEXT,0,None,0
4,4,shift_code,TEXT,0,None,0
5,5,shift_start,TEXT,0,None,0
6,6,shift_end,TEXT,0,None,0
7,7,grace_minutes,INT,0,None,0


2. Actual-in time and minutes after shift start by month


,month,shift_code,records_with_in,earliest_in_time,latest_in_time,avg_minutes_after_shift_start,min_minutes_after_start,max_minutes_after_start
0,2026-04,DAY,2578,05:35:02,15:57:30,41.5,-25.0,597.5
1,2026-04,EVE,2587,13:35:00,23:45:25,-1.7,-25.0,585.4
2,2026-04,GEN,26,08:35:39,09:44:59,-5.4,-24.4,45.0
3,2026-04,NIGHT,2880,05:35:07,23:14:33,37.1,-25.0,502.2
4,2026-05,DAY,2237,05:35:00,15:57:53,57.7,-25.0,597.9
5,2026-05,EVE,2173,13:35:00,22:24:01,-1.6,-25.0,504.0
6,2026-05,GEN,22,08:36:10,09:43:30,-3.7,-23.8,43.5
7,2026-05,NIGHT,2508,05:19:25,23:15:00,61.5,-25.0,561.4
8,2026-06,DAY,2249,05:35:03,15:56:51,65.7,-25.0,596.9
9,2026-06,EVE,2330,13:35:00,22:42:12,-3.4,-25.0,522.2


In [116]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

display(pd.read_sql_query("""
    SELECT
        substr(p.attendance_date, 1, 7) AS month,
        p.shift_code,
        COUNT(*) AS assigned_punches,
        MIN(time(p.punch_time)) AS earliest_punch_time,
        MAX(time(p.punch_time)) AS latest_punch_time,
        ROUND(AVG(
            (julianday(p.punch_time) - julianday(p.shift_start)) * 1440
        ), 1) AS avg_minutes_from_shift_start,
        SUM(CASE WHEN
            (julianday(p.punch_time) - julianday(p.shift_start)) * 1440 < 0
            THEN 1 ELSE 0 END) AS punches_before_shift,
        SUM(CASE WHEN
            (julianday(p.punch_time) - julianday(p.shift_start)) * 1440
            BETWEEN 0 AND 10
            THEN 1 ELSE 0 END) AS punches_within_first_10_min
    FROM stg_punch_shift_assignment p
    JOIN stg_employees_departments e ON e.emp_id = p.emp_id
    WHERE e.plant = 'SITE Plant'
      AND p.attendance_date BETWEEN '2026-04-01' AND '2026-09-30'
    GROUP BY month, p.shift_code
    ORDER BY month, p.shift_code
""", conn))

conn.close()


,month,shift_code,assigned_punches,earliest_punch_time,latest_punch_time,avg_minutes_from_shift_start,punches_before_shift,punches_within_first_10_min
0,2026-04,DAY,4967,05:35:02,15:59:57,248.8,1554,750
1,2026-04,EVE,5394,13:35:00,23:55:00,247.6,1759,747
2,2026-04,GEN,53,08:35:39,17:50:03,239.6,21,6
3,2026-04,NIGHT,5534,04:01:34,23:14:33,245.9,1867,722
4,2026-05,DAY,4240,05:35:00,15:59:26,252.2,1332,586
5,2026-05,EVE,4548,13:35:00,23:54:15,247.2,1501,607
6,2026-05,GEN,48,08:36:10,17:54:27,246.9,15,9
7,2026-05,NIGHT,4736,04:00:51,23:15:00,253.5,1518,616
8,2026-06,DAY,4267,05:35:03,15:59:54,255.3,1251,658
9,2026-06,EVE,4866,13:35:00,23:54:12,246.9,1587,700


In [117]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

for table in ["raw_biometric_punches", "raw_device_mapping"]:
    print(f"\n{table}")
    display(pd.read_sql_query(
        f"PRAGMA table_info({table})", conn
    ))

conn.close()



raw_biometric_punches


,cid,name,type,notnull,dflt_value,pk
0,0,punch_id,TEXT,0,None,0
1,1,device_id,TEXT,0,None,0
2,2,device_user_id,TEXT,0,None,0
3,3,punch_time,TEXT,0,None,0
4,4,_source_file,TEXT,0,None,0
5,5,_loaded_at,TEXT,0,None,0
6,6,_row_hash,TEXT,0,None,0



raw_device_mapping


,cid,name,type,notnull,dflt_value,pk
0,0,device_id,TEXT,0,None,0
1,1,device_user_id,TEXT,0,None,0
2,2,emp_id,TEXT,0,None,0
3,3,enrolled_on,TEXT,0,None,0
4,4,_source_file,TEXT,0,None,0
5,5,_loaded_at,TEXT,0,None,0
6,6,_row_hash,TEXT,0,None,0


In [118]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("peopleops_hackathon.db")

print("1. SITE Plant raw punches by month and device")
display(pd.read_sql_query("""
    SELECT
        substr(p.punch_time, 1, 7) AS month,
        p.device_id,
        COUNT(*) AS raw_punches,
        MIN(time(p.punch_time)) AS earliest_time,
        MAX(time(p.punch_time)) AS latest_time,
        SUM(CASE WHEN time(p.punch_time) BETWEEN '05:30:00'
                  AND '06:10:00' THEN 1 ELSE 0 END) AS around_day_shift_start,
        SUM(CASE WHEN time(p.punch_time) BETWEEN '13:30:00'
                  AND '14:10:00' THEN 1 ELSE 0 END) AS around_eve_shift_start
    FROM raw_biometric_punches p
    JOIN raw_device_mapping m
      ON p.device_id = m.device_id
     AND p.device_user_id = m.device_user_id
    JOIN stg_employees_departments e ON e.emp_id = m.emp_id
    WHERE e.plant = 'SITE Plant'
      AND substr(p.punch_time, 1, 7)
          BETWEEN '2026-04' AND '2026-09'
    GROUP BY month, p.device_id
    ORDER BY month, p.device_id
""", conn))

print("2. Raw punches by month and hour")
display(pd.read_sql_query("""
    SELECT
        substr(p.punch_time, 1, 7) AS month,
        CAST(substr(time(p.punch_time), 1, 2) AS INTEGER) AS punch_hour,
        COUNT(*) AS punch_count
    FROM raw_biometric_punches p
    JOIN raw_device_mapping m
      ON p.device_id = m.device_id
     AND p.device_user_id = m.device_user_id
    JOIN stg_employees_departments e ON e.emp_id = m.emp_id
    WHERE e.plant = 'SITE Plant'
      AND substr(p.punch_time, 1, 7)
          BETWEEN '2026-04' AND '2026-09'
    GROUP BY month, punch_hour
    ORDER BY month, punch_hour
""", conn))

conn.close()


1. SITE Plant raw punches by month and device


,month,device_id,raw_punches,earliest_time,latest_time,around_day_shift_start,around_eve_shift_start
0,2026-04,DEV-SITE-01,16437,04:04:48,23:55:00,3506,3457
1,2026-05,DEV-SITE-01,14131,04:00:51,23:54:15,3060,2927
2,2026-06,DEV-SITE-01,14534,04:02:42,23:54:12,3063,3169
3,2026-07,DEV-SITE-01,16971,00:00:19,23:59:51,24,20
4,2026-08,DEV-SITE-01,15016,00:00:00,23:59:58,18,16
5,2026-09,DEV-SITE-01,16048,00:00:03,23:59:31,18,26


2. Raw punches by month and hour


,month,punch_hour,punch_count
0,2026-04,4,34
1,2026-04,5,1861
2,2026-04,6,3183
3,2026-04,7,148
4,2026-04,8,141
...,...,...,...
94,2026-09,18,82
95,2026-09,20,1
96,2026-09,21,29
97,2026-09,22,1859


In [119]:

import os

base = "final_reports"

print("FINAL REPORTS CHECK\n")

for root, dirs, files in os.walk(base):
    for file in sorted(files):
        path = os.path.join(root, file)
        size_kb = os.path.getsize(path) / 1024
        print(f"✅ {path} ({size_kb:.1f} KB)")

print("\nTotal CSV files:", sum(
    1 for root, dirs, files in os.walk(base)
    for file in files if file.lower().endswith(".csv")
))


FINAL REPORTS CHECK

✅ final_reports\01_monthly_payroll_by_component.csv (2.6 KB)
✅ final_reports\02_overtime_reconciliation.csv (0.4 KB)
✅ final_reports\03_leave_summary.csv (0.3 KB)
✅ final_reports\04_department_payroll.csv (6.0 KB)
✅ final_reports\05_attendance_data_quality.csv (3.9 KB)
✅ final_reports\06_unmatched_punches.csv (348.8 KB)
✅ final_reports\07_unmatched_device_user_exceptions.csv (3.5 KB)
✅ final_reports\08_provisional_daily_attendance.csv (38896.3 KB)
✅ final_reports\09_provisional_attendance_summary.csv (0.1 KB)
✅ final_reports\business_questions\Q01_monthly_payroll_trend.csv (0.2 KB)
✅ final_reports\business_questions\Q02_payroll_by_component.csv (1.9 KB)
✅ final_reports\business_questions\Q03_department_payroll.csv (4.4 KB)
✅ final_reports\business_questions\Q04_overtime_request_status.csv (0.1 KB)
✅ final_reports\business_questions\Q05_monthly_paid_overtime.csv (0.2 KB)
✅ final_reports\business_questions\Q06_top_employees_overtime_cost.csv (0.5 KB)
✅ final_reports\

In [120]:

import os
import zipfile

reports_folder = os.path.abspath("final_reports")
zip_path = os.path.abspath("PeopleOps_HR_Final_Reports.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(reports_folder):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, os.path.dirname(reports_folder))
            zipf.write(full_path, arcname)

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))
print("Exists:", os.path.exists(zip_path))


ZIP created: C:\Users\dell\Desktop\data-hackathon-main\PeopleOps_HR_Final_Reports.zip
ZIP size (MB): 3.92
Exists: True
